# PerFed-SKD on VeReMi NextGen — 100 edge devices

Singh, Rupchandani & Adhikari, *Personalized Federated Learning for Heterogeneous Edge
Device: Self-Knowledge Distillation Approach* — Eq. (2)–(3), Algorithm 1 (server) and
Algorithm 2 (device), with **DAGSNet** (395,024 params) as every model in the system.
The paper's server-side teacher trained on a "predefined dataset" and any feature
extraction backbone are deliberately absent: the only knowledge transfer here is the one
the method is named after, from a device's own previous personalized model to its
current one.

| from the paper | from `knowledge/` and the owner's decisions |
|---|---|
| Eq. (2) φ_m(ω) = f_m(ω) + λ·L( x(V_m) ‖ x(ω) ); V_m = the device's model from the previous round, frozen | L = KL(p_teacher ‖ p_student) on softmax, temperature 1, λ = 1.0; teacher in `eval()` under `no_grad` |
| Alg. 2 l.4–7: selected devices take ω^t, the rest keep their own weights; **all** devices train | one local epoch, batch 256 |
| Alg. 1 l.10: ω^{t+1} = (1/\|S\|) Σ_{m∈S} ω_m^{t+1} | only the selected devices upload |
| Alg. 1 l.11 + l.4–6: A ← (1/\|M\|) Σ a_m, τ = A, select m where a_m < τ | a_m = **accuracy** of device m's model on the full global test set |
| Eq. (3): "Stochastic Gradient Descent", no rate given | AdamW, wd 0.0001, re-created per device per round; lr 0.001 → 1e-05 cosine per round; clip 1.0; fp16 AMP; seed 42 |

**Round 1** selects every device (no accuracies exist yet) and every V_m is ω^0, so the
distillation term starts at exactly 0 and the teacher becomes a genuinely different model
from round 2 on.

**Evaluation:** every round, every device's personalized model on the full
10,761,343-row test set — 10 metrics per device, mean/std/min/max over all 100 — plus the
server's aggregate ω^t as one extra model (`global_*` columns, never mixed into the
device mean). The same numbers feed the threshold, so the selection rule is measured on
exactly the artifacts that are published.

**Deviations, all deliberate** (full list in `rebuild.md` §2): no server-side teacher and
no backbone; τ is Algorithm 1's mean of device accuracies, not the prose's "global model
accuracy" (which on this data would select every device in every round); all devices
train and only the selected ones upload, following §IV's prose over Algorithm 1's loop
bound; KL at temperature 1 with λ = 1.0 where the paper gives no divergence and no λ;
AdamW with a per-round cosine schedule where the paper gives neither; the global test set
is shared by every device, so these scores measure global generalization of a
personalized model, not performance on the device's own distribution.

Per-round output is **weights only** (ω^t plus every ω_m^t as `state_dict` tensors, with
the selected subset and τ in the same file); `proj/ckpt.py::load_weights` rebuilds them at
any round, and the last cell re-derives every published metric — including τ and the
selection — from the confusion matrices on disk.


In [ ]:
import os, subprocess, sys, time, torch
T0 = time.monotonic()     # session clock: the 12 h cap charges for spawn and compile too.
n = torch.cuda.device_count()
assert n == 2, f"expected 2 GPUs, got {n}. machine_shape must be NvidiaTeslaT4."
for i in range(n):
    cap = torch.cuda.get_device_capability(i)
    assert cap == (7, 5), f"GPU {i} is {cap}, expected (7,5) Tesla T4"
    print(i, torch.cuda.get_device_name(i), cap,
          f"{torch.cuda.get_device_properties(i).total_memory/2**30:.1f} GiB")
print("torch", torch.__version__, "| python", sys.version.split()[0])
# NCCL is unused (FL clients never form a process group) but P2P probing can still hang.
os.environ["NCCL_P2P_DISABLE"] = "1"; os.environ["NCCL_IB_DISABLE"] = "1"
os.environ["TORCHINDUCTOR_COMPILE_THREADS"] = "1"
os.makedirs("/kaggle/working/proj", exist_ok=True)
open("/kaggle/working/proj/__init__.py", "w").close()
sys.path.insert(0, "/kaggle/working")


In [ ]:
CFG = dict(
    # --- architecture: knowledge/architecture.md, frozen (395,024 params, every model)
    patch_len=6, stem_ch=96, dense_growth=32, dense_layers=3,
    incep_modules=2, fire_modules=3, dropout=0.1,
    num_classes=16, n_features=66,
    # --- PerFed-SKD: owner's decisions 2026-09-21 (the paper gives no optimizer, no rate,
    #     no lambda and no divergence family); LR cosine per round lr -> lr_min
    #     (proj/perfedskd.py::lr_at). `select_metric` only DECLARES the rule so the
    #     fingerprint can see it -- proj/perfedskd.py::SELECT_METRIC is the rule, and the
    #     driver refuses to start if the two disagree.
    lam=1.0, select_metric="accuracy",
    lr=0.001, lr_schedule="cosine", lr_min=1e-05,
    weight_decay=0.0001, local_epochs=1,
    rounds=2, clip=1.0, seed=42,
    # --- compute: knowledge/dataset.md §4
    n_clients=100, batch=256, eval_batch=16384,
    device="cuda", world_size=2, compile=True,
    # Each client starts a fresh GradScaler at 2**16 and spends a few steps calibrating.
    # Fixed before the first measurement so it cannot be widened afterwards.
    max_skips_per_client=16,
    preds_rounds=[],   # (N, 10.76 M) uint8 per listed round
    finalize_reserve_seconds=900,   # never start a round that leaves no time to commit it
    run_name="perfedskd_100c_probe",
    cache="/kaggle/temp/veremi_cache",
    max_seconds=2.5 * 3600,   # 12 h hard cap; leave room to finalize artifacts
    require_resume=False,
)
# data_id is filled in below, once the feature order and scaler are known. It is part of
# proj/ckpt.py FINGERPRINT_KEYS, so a run cannot resume across a changed preprocessing.
for k, v in CFG.items(): print(f"{k:>20} = {v}")


In [ ]:
%%writefile /kaggle/working/proj/model.py
"""DAGSNet — Khan et al. 2025 §4.10, Eq. (38)-(48). 395.024 tham số.
Vào: (B, 66) đặc trưng đã z-score.  Ra: (B, 16) logit (CHƯA softmax).

The lightweight-FL NILM rebuild uses this one class for every model in the system: each
client's personalized model w_s, each client's proxy w_r and the server's aggregate
w_bar_r are all DAGSNet classifiers with 16 logits, built by `build_model` and started
from the SAME seeded initialization (owner's decision; the paper searches a personalized
architecture per device with MNAS, which is deliberately left out here).
"""
import torch
import torch.nn as nn


def cbr(i, o, k):
    """Conv → BatchNorm → ReLU. bias=False vì BatchNorm ngay sau đã có tham số dịch."""
    return nn.Sequential(nn.Conv1d(i, o, k, padding=k // 2, bias=False),
                         nn.BatchNorm1d(o), nn.ReLU(inplace=True))


class DenseNet1d(nn.Module):
    """Eq. (38)-(39): mỗi lớp nhận nối của toàn bộ feature map trước đó."""
    def __init__(self, cin, growth, layers):
        super().__init__()
        self.blocks = nn.ModuleList([cbr(cin + i * growth, growth, 3) for i in range(layers)])
        self.out_ch = cin + layers * growth

    def forward(self, x):
        for b in self.blocks:
            x = torch.cat([x, b(x)], dim=1)                     # Eq. (38)
        return x                                                # Eq. (39)


class Inception1d(nn.Module):
    """Eq. (40): bốn nhánh song song 1x1 / 3x3 / 5x5 / pool, nối lại."""
    def __init__(self, cin, c):
        super().__init__()
        self.b1 = cbr(cin, c, 1)
        self.b3 = nn.Sequential(cbr(cin, c, 1), cbr(c, c, 3))
        self.b5 = nn.Sequential(cbr(cin, c, 1), cbr(c, c, 5))
        self.bp = nn.Sequential(nn.MaxPool1d(3, 1, 1), cbr(cin, c, 1))
        self.out_ch = 4 * c

    def forward(self, x):
        return torch.cat([self.b1(x), self.b3(x), self.b5(x), self.bp(x)], dim=1)


class GoogleNet1d(nn.Module):
    """Eq. (40)-(41): các inception module xếp chồng."""
    def __init__(self, cin, modules_n, c=32):
        super().__init__()
        mods, ch = [], cin
        for _ in range(modules_n):
            m = Inception1d(ch, c); mods.append(m); ch = m.out_ch
        self.net = nn.Sequential(*mods); self.out_ch = ch

    def forward(self, x):
        return self.net(x)


class AlexNet1d(nn.Module):
    """Eq. (42)-(44). ceil_mode=True: trục vị trí chỉ dài 11, không được để pool co về 0."""
    def __init__(self, cin, ch=128):
        super().__init__()
        self.net = nn.Sequential(
            cbr(cin, ch, 3), nn.MaxPool1d(2, ceil_mode=True),
            cbr(ch, ch, 3),  nn.MaxPool1d(2, ceil_mode=True),
            cbr(ch, ch, 3))
        self.out_ch = ch

    def forward(self, x):
        return self.net(x)


class Fire1d(nn.Module):
    """Eq. (45)-(46): squeeze 1x1 nuôi hai nhánh expand 1x1 và 3x3."""
    def __init__(self, cin, sq, ex):
        super().__init__()
        self.squeeze = cbr(cin, sq, 1)                          # Eq. (46)
        self.e1 = cbr(sq, ex, 1)
        self.e3 = cbr(sq, ex, 3)
        self.out_ch = 2 * ex

    def forward(self, x):
        s = self.squeeze(x)
        return torch.cat([self.e1(s), self.e3(s)], dim=1)       # Eq. (45)


class SqueezeNet1d(nn.Module):
    def __init__(self, cin, modules_n, sq=32, ex=48):
        super().__init__()
        mods, ch = [], cin
        for _ in range(modules_n):
            m = Fire1d(ch, sq, ex); mods.append(m); ch = m.out_ch
        self.net = nn.Sequential(*mods); self.out_ch = ch

    def forward(self, x):
        return self.net(x)


class DAGSNet(nn.Module):
    def __init__(self, cfg, n_features, out_dim):
        super().__init__()
        self.patch_len = cfg["patch_len"]
        assert n_features % self.patch_len == 0
        self.k = n_features // self.patch_len                   # 11
        cin = self.patch_len                                    # 6 kênh

        s = cfg["stem_ch"]
        self.stems   = nn.ModuleList([cbr(cin, s, 1) for _ in range(4)])
        self.dense   = DenseNet1d(s, cfg["dense_growth"], cfg["dense_layers"])
        self.google  = GoogleNet1d(s, cfg["incep_modules"])
        self.alex    = AlexNet1d(s)
        self.squeeze = SqueezeNet1d(s, cfg["fire_modules"])
        comb = self.dense.out_ch + self.google.out_ch + self.alex.out_ch + self.squeeze.out_ch

        self.head = nn.Sequential(                              # Eq. (48)
            nn.LayerNorm(comb), nn.Dropout(cfg["dropout"]),
            nn.Linear(comb, 256), nn.ReLU(inplace=True),
            nn.Dropout(cfg["dropout"]), nn.Linear(256, out_dim))

    def forward(self, x):                                       # (B, n_features)
        # view rồi MỚI transpose: gom 6 cột liên tiếp thành một patch, sau đó patch
        # mới trở thành trục vị trí. Làm view(B, 6, 11) thẳng sẽ trộn sai các cột.
        Fm = x.view(x.shape[0], self.k, self.patch_len).transpose(1, 2)   # (B, 6, 11)
        feats = [gp(stem(Fm)) for stem, gp in
                 zip(self.stems, [self.dense, self.google, self.alex, self.squeeze])]
        pooled = [f.mean(dim=-1) for f in feats]                # global average pool
        return self.head(torch.cat(pooled, dim=1))              # Eq. (47) -> (48)


# Cấu hình ĐÚNG như knowledge/architecture.md. Đổi bất kỳ giá trị nào ở đây thì
# state_dict sẽ không nạp được — đó là chủ ý.
CFG = {
    "patch_len": 6, "stem_ch": 96, "dense_growth": 32, "dense_layers": 3,
    "incep_modules": 2, "fire_modules": 3, "dropout": 0.1, "num_classes": 16,
}
N_PARAMS = 395_024            # every model in the system: head 256 -> 16


def build_model(cfg):
    """One DAGSNet classifier. Used for the personalized model w_s, the proxy w_r and the
    aggregate w_bar_r alike. cfg is the dict saved in the checkpoint, so a model rebuilt
    here matches the one that produced those weights."""
    return DAGSNet({k: cfg[k] for k in CFG}, n_features=cfg["n_features"],
                   out_dim=cfg["num_classes"])


In [ ]:
%%writefile /kaggle/working/proj/ckpt.py
"""Weights, resume state and the completion marker — three files, one atomic round.

The per-round weights file holds WEIGHTS ONLY and loads with weights_only=True: the
server's aggregate omega^t and every client's personalized model omega_m^t as plain
state_dict tensors. RNG and the round counter live in a separate resume bundle keyed by
the same round, so a reader never has to unpickle arbitrary objects to look at a
checkpoint.

Persistent state of PerFed-SKD is: the round, omega^t, the M personalized models (each of
which is BOTH the client's current model and next round's teacher V_m), and the device
subset S_{t+1} the server selected from this round's accuracies. `selected_next` is
therefore part of the checkpoint, not a derived convenience: without it a resumed session
would have to re-derive the selection, and a run that resumed would not be the run that
was interrupted. AdamW is re-created per client per round (owner's decision, see
perfedskd.py), so no optimizer state exists at a round boundary.
"""
import csv, hashlib, json, os, random, shutil
from pathlib import Path
import numpy as np, torch

SUBDIRS = ("weights", "resume", "complete", "metrics", "preds", "confusion", "reports", "logs")

# What a later session imports. `preds` and `logs` are not needed to CONTINUE, but a run
# split across sessions must still be verifiable from its final output alone.
RESUME_SUBDIRS = ("weights", "resume", "metrics", "confusion", "preds", "logs")

# A run tree may start at a round r0 > 1 when the previous sessions were handed over as a
# HANDOFF BUNDLE: only round r0's artifacts, plus this file, which carries the sha256 of
# every weights file 1..r0 taken from the full tree the owner holds locally. Round r0 is
# then anchored two ways — its own bytes must hash to chain[r0], and its prev_sha must be
# chain[r0-1] — so a bundle cannot be assembled from two runs of the same config, and the
# full chain is re-verified locally after the sessions are merged (merge_sessions.py).
HANDOFF = "reports/handoff.json"

# Every input that changes what the numbers mean. `rounds` IS in the list since the
# per-round LR schedule (proj.perfedskd.lr_at) spans the whole run: the weights at round r
# depend on how many rounds were planned, so a continuation push must plan the same
# total. `lam` and `select_metric` are here because they change the objective and the
# device-selection rule respectively -- two runs that differ in either are not the same
# experiment. Paths, world_size, compile, eval_batch, max_seconds and require_resume are
# operational and absent.
FINGERPRINT_KEYS = (
    "patch_len", "stem_ch", "dense_growth", "dense_layers", "incep_modules",
    "fire_modules", "dropout", "num_classes", "n_features",
    "lr", "lr_schedule", "lr_min", "rounds", "weight_decay", "clip",
    "lam", "select_metric",
    "n_clients", "batch", "local_epochs", "seed",
    "data_id", "run_name",
)


def run_dir(run_name, base="/kaggle/working/runs"):
    d = Path(base) / run_name
    for s in SUBDIRS: (d / s).mkdir(parents=True, exist_ok=True)
    return d


def fingerprint(cfg):
    """A missing key is a KeyError, never a default. Silently hashing `None` for a key that
    was renamed is exactly how a fingerprint stops protecting anything."""
    missing = [k for k in FINGERPRINT_KEYS if k not in cfg]
    if missing:
        raise KeyError(f"fingerprint needs {missing} in CFG; add them, do not default them")
    payload = json.dumps({k: cfg[k] for k in FINGERPRINT_KEYS}, sort_keys=True, default=str)
    return hashlib.sha256(payload.encode()).hexdigest()[:16]


def file_sha(path):
    """Hash of the file as written. Not a re-serialisation: torch.save embeds a zip whose
    bytes are not reproducible, so only the bytes on disk are a stable identity."""
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def atomic_save(obj, path):
    path = Path(path); tmp = path.with_suffix(path.suffix + ".tmp")
    torch.save(obj, tmp); os.replace(tmp, path)     # replace is atomic on POSIX


def atomic_np_save(path, arr):
    """np.save appends .npy to a name that lacks it, so write through a handle."""
    path = Path(path); tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "wb") as f:
        np.save(f, arr); f.flush(); os.fsync(f.fileno())
    os.replace(tmp, path)


# --- RNG kept as tensors and primitives so the resume bundle also loads weights_only=True.
def rng_state():
    npy = np.random.get_state()
    return {"python": random.getstate(),
            "numpy": (npy[0], torch.from_numpy(npy[1].copy()), int(npy[2]), int(npy[3]),
                      float(npy[4])),
            "torch": torch.get_rng_state(),
            "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None}


def set_rng_state(s):
    random.setstate(tuple(s["python"]))
    n = s["numpy"]
    np.random.set_state((n[0], n[1].numpy().astype(np.uint32), n[2], n[3], n[4]))
    torch.set_rng_state(s["torch"].cpu())
    if s.get("cuda") is not None: torch.cuda.set_rng_state_all(s["cuda"])


def cpu_sd(model):
    """A detached CPU copy of a module's state_dict: tensors only, so the file it goes
    into loads with weights_only=True."""
    core = getattr(model, "_orig_mod", model)                 # unwrap torch.compile
    return {k: v.detach().cpu().clone() for k, v in core.state_dict().items()}


# --------------------------------------------------------------------------- write
def save_round_weights(global_sd, clients_sd, rnd, cfg, metrics, global_metrics, d,
                       selected, selected_next, tau):
    """global_sd: state_dict of omega^t; clients_sd: {cid: state_dict of omega_m^t} for
    ALL M clients. `metrics` is the mean over clients of the 10 metrics,
    `global_metrics` the 10 metrics of omega^t. `selected` is the subset S_t that was
    aggregated into omega^t, `selected_next` the subset S_{t+1} chosen from this round's
    accuracies, and `tau` the threshold that produced it. The caller writes the marker,
    and only after every other artifact is on disk."""
    prev = d / "weights" / f"round_{rnd - 1:03d}.pt"
    # The hash of the weights this round was trained FROM. Two runs of the same config have
    # the same fingerprint, so without this an import can keep round 1 of run A and take
    # round 2 of run B and call the result one training history.
    prev_sha = file_sha(prev) if rnd > 1 and prev.is_file() else None
    atomic_save({"round": int(rnd),
                 "global": global_sd,
                 "clients": {int(c): sd for c, sd in sorted(clients_sd.items())},
                 "cfg": {k: v for k, v in cfg.items()},       # rebuild recipe
                 "fingerprint": fingerprint(cfg),
                 "prev_sha": prev_sha,
                 "metrics": metrics,
                 "global_metrics": global_metrics,
                 # The algorithm state a continuation needs, in the same atomic unit as
                 # the weights it belongs to. Plain ints: weights_only=True must still load.
                 "selected": [int(c) for c in selected],
                 "selected_next": [int(c) for c in selected_next],
                 "tau": float(tau)},
                d / "weights" / f"round_{rnd:03d}.pt")
    # The RNG here is the DRIVER's, and the driver does not train. Recorded for forensics:
    # worker training RNG is re-derived from (seed, round, client), so continuation does
    # not depend on restoring this.
    atomic_save({"round": int(rnd), "rng": rng_state(),
                 "selected_next": [int(c) for c in selected_next],
                 "note": "no optimizer state: AdamW is re-created per client per round; "
                         "worker RNG derives from (seed, round, client)",
                 "fingerprint": fingerprint(cfg)},
                d / "resume" / f"round_{rnd:03d}.pt")
    return d / "weights" / f"round_{rnd:03d}.pt"


def mark_complete(d, rnd):
    (d / "complete" / f"round_{rnd:03d}.done").write_text("")


# --------------------------------------------------------------------------- rebuild
def _load_into(model, sd, expect_params):
    sd = {k.removeprefix("module.").removeprefix("_orig_mod."): v for k, v in sd.items()}
    bad = [k for k, v in sd.items() if v.is_floating_point() and not torch.isfinite(v).all()]
    if bad:
        raise RuntimeError(f"non-finite values in checkpoint tensors {bad[:3]}")
    model.load_state_dict(sd, strict=True)
    n = sum(p.numel() for p in model.parameters())
    if expect_params is not None and n != expect_params:
        raise RuntimeError(f"rebuilt {n:,} parameters, expected {expect_params:,}")
    return model.eval()


def load_weights(path, build_model, expect_params=None, clients=None, device="cpu"):
    """Rebuild omega^t and the personalized models at exactly this checkpoint, from
    weights alone. Returns (G, {cid: M_m}, raw dict); the raw dict also carries
    `selected_next`, the device subset the next round must start from.

    Every assert here exists because its failure is otherwise silent: strict=True catches a
    filtered running_mean/var (eval() would then normalize by 0/1 and report nothing), the
    parameter count catches a cfg that drifted from the one that trained these weights, and
    the finite check catches a diverged tensor that argmax would turn into a plausible label.
    `clients=None` rebuilds every client; pass a list to rebuild a subset."""
    ck = torch.load(path, map_location="cpu", weights_only=True)
    if ck.get("fingerprint") != fingerprint(ck["cfg"]):
        raise RuntimeError("weights file fingerprint disagrees with its own cfg")
    G = _load_into(build_model(ck["cfg"]), ck["global"], expect_params).to(device)
    want = sorted(ck["clients"]) if clients is None else list(clients)
    if sorted(ck["clients"]) != list(range(ck["cfg"]["n_clients"])):
        raise RuntimeError(f"checkpoint holds clients {sorted(ck['clients'])[:5]}..., "
                           f"expected 0..{ck['cfg']['n_clients'] - 1}")
    Ms = {int(c): _load_into(build_model(ck["cfg"]), ck["clients"][c], expect_params).to(device)
          for c in want}
    sel = ck.get("selected_next")
    if sel is None or not all(isinstance(c, int) and 0 <= c < ck["cfg"]["n_clients"]
                              for c in sel):
        raise RuntimeError(f"checkpoint has no usable selected_next ({sel!r}); a resumed "
                           "run would have to invent the device subset for the next round")
    return G, Ms, ck


# --------------------------------------------------------------------------- verify
def read_handoff(d):
    """The handoff record of a tree that starts past round 1, or None. A malformed file
    reads as absent, and an absent record means rounds before the first marker are simply
    missing — which last_complete_round then treats as a gap, never as a start."""
    p = Path(d) / HANDOFF
    if not p.is_file():
        return None
    try:
        h = json.loads(p.read_text())
        h["round"] = int(h["round"])
        h["chain"] = {int(k): str(v) for k, v in h["chain"].items()}
        return h
    except Exception:
        return None


def handoff_record(d, rnd, fp, extra=None):
    """The sha chain 1..rnd of a FULL tree, for a bundle that will carry round rnd alone.
    (The selection state travels inside the round's own weights file, so a bundle of one
    round is still a complete resume point.)
    Refuses a tree whose chain does not verify all the way from round 1. The caller writes
    the dict to HANDOFF inside the bundle, never inside the full tree."""
    d = Path(d)
    if _first_round(d) != 1:
        raise RuntimeError(f"{d} is itself a partial tree; export a handoff from the merged run")
    last = last_complete_round(d, fp)
    if last is None or last < rnd:
        raise RuntimeError(f"{d}: rounds 1..{rnd} do not all verify (last={last}); "
                           "nothing to hand off")
    chain = {str(r): file_sha(d / "weights" / f"round_{r:03d}.pt") for r in range(1, rnd + 1)}
    return {"round": int(rnd), "fingerprint": fp, "chain": chain, **(extra or {})}


def round_ok(d, rnd, fp=None, chain=True):
    """A round counts only if every artifact of that round is present, READABLE, and links
    to the round before it. The marker alone proves nothing. When the round before it is
    not on disk, the link is checked against the handoff chain instead (see HANDOFF), and
    the round's own bytes must match the chain too."""
    w = d / "weights" / f"round_{rnd:03d}.pt"
    r = d / "resume" / f"round_{rnd:03d}.pt"
    m = d / "metrics" / f"round_{rnd:03d}.json"
    c = d / "confusion" / f"round_{rnd:03d}.npy"
    g = d / "confusion" / f"global_{rnd:03d}.npy"
    for path in (w, r, m, c, g):
        if not path.is_file() or path.stat().st_size == 0:
            return False
    try:
        ck = torch.load(w, map_location="cpu", weights_only=True, mmap=True)
        rs = torch.load(r, map_location="cpu", weights_only=True)   # not just "it exists"
        row = json.loads(m.read_text())
        np.load(c); np.load(g)
    except Exception:
        return False                      # truncated or corrupt reads as a failed round
    if int(ck.get("round", -1)) != rnd or int(row.get("round", -1)) != rnd:
        return False
    if int(rs.get("round", -1)) != rnd:
        return False
    if fp is not None and (ck.get("fingerprint") != fp or rs.get("fingerprint") != fp):
        return False
    if chain:
        # Round r is only meaningful as the product of round r-1. Same config, same
        # fingerprint, different training history -> different bytes -> chain breaks here.
        prev = d / "weights" / f"round_{rnd - 1:03d}.pt"
        if rnd > 1 and not prev.is_file():
            h = read_handoff(d)
            if h is None or h["round"] != rnd or (fp is not None and h["fingerprint"] != fp):
                return False                  # a bare round r > 1 is a gap, not a start
            if h["chain"].get(rnd) != file_sha(w) or ck.get("prev_sha") != h["chain"].get(rnd - 1):
                return False
        else:
            want = file_sha(prev) if rnd > 1 and prev.is_file() else None
            if ck.get("prev_sha") != want:
                return False
    return True


def _markers(d):
    return sorted(int(p.stem.split("_")[1]) for p in (d / "complete").glob("round_*.done"))


def _first_round(d):
    """1 for a full tree; the handoff round for a bundle that starts later; the lowest
    marker otherwise (round_ok then rejects it as a gap unless a handoff anchors it)."""
    h = read_handoff(d)
    m = _markers(d)
    if h is not None and not (d / "weights" / f"round_{h['round'] - 1:03d}.pt").is_file():
        return h["round"]
    return m[0] if m else 1


def last_complete_round(d, fp=None):
    """Largest r such that rounds first..r are ALL complete, where first is 1 or the
    handoff round of a bundle. A gap ends the run."""
    last = 0
    m = _markers(d)
    top = m[-1] if m else 0
    for r in range(_first_round(d), top + 1):
        if not (d / "complete" / f"round_{r:03d}.done").is_file() or not round_ok(d, r, fp):
            break
        last = r
    return last or None


# --------------------------------------------------------------------------- resume
def _attached_source(run_name, attached=Path("/kaggle/input")):
    if not attached.exists():
        return None
    roots = sorted({p.parent for p in attached.rglob("complete/round_*.done")
                    if run_name in p.parts})
    if len(roots) > 1:
        raise RuntimeError(f"Multiple resume trees for {run_name}: {roots}")
    return roots[0].parent if roots else None


def _import_from(src, d, fp):
    """Copy through a staging tree, verify there, then publish one round at a time with its
    marker last. Idempotent: a crash mid-publish leaves that round unmarked, and the next
    attempt re-copies it from the still-mounted source. The source's own markers bound the
    import, and the destination must not already hold a different history."""
    stage = d.parent / f".{d.name}.import"
    shutil.rmtree(stage, ignore_errors=True)
    for sub in RESUME_SUBDIRS + ("complete", "reports"):
        (stage / sub).mkdir(parents=True, exist_ok=True)
    for sub in RESUME_SUBDIRS + ("complete",):
        peer = src / sub
        if not peer.is_dir(): continue
        for f in peer.iterdir():
            if f.is_file():
                # copyfile, not copy2: a read-only mount's mode would carry across and the
                # first rewrite would die with PermissionError.
                shutil.copyfile(f, stage / sub / f.name)
                os.chmod(stage / sub / f.name, 0o644)
    if (src / HANDOFF).is_file():
        shutil.copyfile(src / HANDOFF, stage / HANDOFF)
        os.chmod(stage / HANDOFF, 0o644)

    first = _first_round(stage)
    src_last = first - 1
    while ((stage / "complete" / f"round_{src_last + 1:03d}.done").is_file()
           and round_ok(stage, src_last + 1, fp)):
        src_last += 1
    if src_last < first:
        src_last = 0                                   # nothing verifiable, not even round first

    if src_last and first > 1:
        # The destination must not hold rounds the bundle cannot vouch for.
        if any((d / "weights" / f"round_{r:03d}.pt").is_file() for r in range(1, first)):
            shutil.rmtree(stage, ignore_errors=True)
            raise RuntimeError(f"{d} already holds rounds before {first}; a handoff bundle "
                               "cannot be spliced onto a tree that has its own history")
        shutil.copyfile(stage / HANDOFF, d / HANDOFF)
    for r in range(first, src_last + 1):
        here = d / "weights" / f"round_{r:03d}.pt"
        if here.is_file() and file_sha(here) != file_sha(stage / "weights" / f"round_{r:03d}.pt"):
            shutil.rmtree(stage, ignore_errors=True)
            raise RuntimeError(
                f"round {r} in {d} and in {src} have the same config but different weights: "
                "these are two different training runs, not one interrupted one. Refusing to "
                "splice them. Detach one source, or start a new run_name.")

    published = 0
    for r in range(first, src_last + 1):
        if not round_ok(d, r, fp):
            for sub in RESUME_SUBDIRS:
                for f in list((stage / sub).glob(f"round_{r:03d}.*")) + \
                         list((stage / sub).glob(f"global_{r:03d}.*")):
                    shutil.copyfile(f, d / sub / f.name)
                    os.chmod(d / sub / f.name, 0o644)
        if not (d / "complete" / f"round_{r:03d}.done").is_file():
            mark_complete(d, r)                     # marker last, per round
        published = r
    shutil.rmtree(stage, ignore_errors=True)
    n_mark = len(list((src / "complete").glob("round_*.done"))) if (src / "complete").is_dir() else 0
    print(f"[resume] {src}: {n_mark} marker(s), verified to round {src_last}, imported {first}..{published}"
          + (f" (handoff bundle: chain 1..{first} attested, verified locally before export)"
             if first > 1 else "")
          if published else
          f"[resume] {src} held no verifiable committed round; starting from 0")
    return published


def resolve_resume(run_name, cfg=None, attached=Path("/kaggle/input"), d=None):
    """working/ first, then any attached input (previous kernel output or a checkpoint
    dataset). Returns the last round that is complete AND verified, or None."""
    d = run_dir(run_name) if d is None else d
    fp = fingerprint(cfg) if cfg is not None else None
    src = _attached_source(run_name, attached)
    if src is not None:
        _import_from(src, d, fp)
    rebuild_history(d, fp)
    return last_complete_round(d, fp)


# --------------------------------------------------------------------------- history
def _write_csv(p, rows):
    tmp = Path(p).with_suffix(".csv.tmp")
    with open(tmp, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0]))
        w.writeheader(); w.writerows(rows)
        f.flush(); os.fsync(f.fileno())
    os.replace(tmp, p)                      # a crash mid-write cannot truncate the live file


def rebuild_history(d, fp=None):
    """history.csv (mean over clients + global proxy per round) and clients.csv (one row
    per client per round) are DERIVED from metrics/round_NNN.json, never authoritative.
    Only the verified contiguous range 1..last goes in."""
    last = last_complete_round(d, fp) or 0
    rows, crow = [], []
    for r in range(_first_round(d), last + 1):
        p = d / "metrics" / f"round_{r:03d}.json"
        if not p.is_file(): break
        try: j = json.loads(p.read_text())
        except Exception: break
        # `selected`/`selected_next` are LISTS: they belong in the per-round json, not in a
        # CSV cell. The scalar summary (n_selected, n_selected_next, tau) is already in the
        # row, and append_history never wrote the lists -- so including them here would make
        # a rebuilt history.csv disagree, column for column, with an appended one.
        rows.append({k: v for k, v in j.items()
                     if k not in ("clients", "global", "selected", "selected_next")})
        crow += [{"round": r, **{k: v for k, v in c.items() if k != "per_class"}}
                 for c in j["clients"]]
    if rows:
        _write_csv(d / "history.csv", rows)
        _write_csv(d / "clients.csv", crow)
    else:
        for n in ("history.csv", "clients.csv"):
            if (d / n).exists(): (d / n).unlink()  # a stale CSV outlives the rounds it described
    return len(rows)


def append_history(d, row, client_rows):
    """Keyed by round: a redone round replaces its lines instead of duplicating them."""
    p = d / "history.csv"
    rows = {}
    if p.exists():
        with open(p) as f:
            rows = {int(r["round"]): r for r in csv.DictReader(f)}
    rows[int(row["round"])] = {k: str(v) for k, v in row.items()}
    _write_csv(p, [rows[k] for k in sorted(rows)])
    q = d / "clients.csv"
    old = []
    if q.exists():
        with open(q) as f:
            old = [r for r in csv.DictReader(f) if int(r["round"]) != int(row["round"])]
    _write_csv(q, old + [{k: str(v) for k, v in c.items()} for c in client_rows])


In [ ]:
%%writefile /kaggle/working/proj/metrics.py
"""All 10 metrics from a full confusion matrix. No batch averaging, no sampling."""
import numpy as np

METRIC_KEYS = ("accuracy", "precision_macro", "precision_micro", "precision_weighted",
               "recall_macro", "recall_micro", "recall_weighted",
               "f1_macro", "f1_micro", "f1_weighted")


def metrics_from_confusion(cm):
    """cm[i, j] = count of true class i predicted as j. Integer counts in, 10 floats out."""
    cm = np.asarray(cm, dtype=np.float64)
    tp = np.diag(cm)
    support = cm.sum(axis=1)                       # true count per class
    pred = cm.sum(axis=0)                          # predicted count per class
    total = cm.sum()

    # A class never predicted has precision 0/0; sklearn defines it as 0 with zero_division=0.
    prec = np.divide(tp, pred, out=np.zeros_like(tp), where=pred > 0)
    rec = np.divide(tp, support, out=np.zeros_like(tp), where=support > 0)
    denom = prec + rec
    f1 = np.divide(2 * prec * rec, denom, out=np.zeros_like(tp), where=denom > 0)

    acc = tp.sum() / total
    w = support / total                            # weighted = support-weighted mean
    # float(), not np.float64: a numpy scalar anywhere in a checkpoint dict makes
    # torch.load(weights_only=True) refuse the whole file, and the failure only appears
    # when something later tries to read it back.
    return {k: float(v) for k, v in (
        ("accuracy", acc),
        ("precision_macro", prec.mean()), ("precision_micro", acc),
        ("precision_weighted", (prec * w).sum()),
        ("recall_macro", rec.mean()), ("recall_micro", acc),
        ("recall_weighted", (rec * w).sum()),
        ("f1_macro", f1.mean()), ("f1_micro", acc),
        ("f1_weighted", (f1 * w).sum()))}


def per_class_from_confusion(cm, class_names):
    cm = np.asarray(cm, dtype=np.float64)
    tp, support, pred = np.diag(cm), cm.sum(axis=1), cm.sum(axis=0)
    prec = np.divide(tp, pred, out=np.zeros_like(tp), where=pred > 0)
    rec = np.divide(tp, support, out=np.zeros_like(tp), where=support > 0)
    d = prec + rec
    f1 = np.divide(2 * prec * rec, d, out=np.zeros_like(tp), where=d > 0)
    return [{"idx": i, "class": class_names[i], "support": int(support[i]),
             "precision": float(prec[i]), "recall": float(rec[i]), "f1": float(f1[i])}
            for i in range(len(class_names))]


In [ ]:
%%writefile /kaggle/working/proj/data.py
"""Parquet -> resident fp16 tensors. One pass per session, then no input pipeline at all.

Two facts from knowledge/dataset.md that produce silent corruption if ignored:
  * train/ is ALREADY z-scored; test/ is NOT. Applying scaler.json to train a second time
    destroys it and raises nothing.
  * the integer label is in column `label` (int8, 0..15). Decoding `attack_type` strings
    over 43 M rows costs tens of seconds per pass for the same information.
"""
import json
from pathlib import Path
import numpy as np
import pyarrow.dataset as ds

CACHE_FILES = ("train_X.f16.npy", "train_y.u8.npy", "test_X.f16.npy",
               "test_y.u8.npy", "spans.json")


def find_root(sentinel, bases=("/kaggle/input",)):
    """Kaggle mounts are nested by kind and owner; the prefix is not /kaggle/input/<slug>/.
    Resolve by locating the sentinel instead of hard-coding a depth."""
    depth = len(Path(sentinel).parts)          # strip the whole sentinel, not one level
    hits = []
    for b in bases:
        p = Path(b)
        if p.exists():
            for q in p.rglob(sentinel):
                if q.is_dir():
                    r = q
                    for _ in range(depth): r = r.parent
                    hits.append(r)
    hits = sorted(set(hits))
    if len(hits) != 1:
        raise RuntimeError(f"expected exactly one {sentinel!r} under {bases}, got {hits}")
    return hits[0]


def parquet_files(root):
    """The parquet parts of a directory, in a fixed order, and NOTHING else.

    `ds.dataset(dir, format="parquet")` opens every file it finds. The centralized test
    directory ships a `part-NNNNN.stats.json` sidecar next to each part, so `to_table()`
    died on Kaggle with "Parquet magic bytes not found in footer" after a nine-minute train
    decode. It survived locally only because the smoke fixture reads through `to_batches()`
    and breaks early, never reaching a sidecar -- a lazy reader hides exactly this.

    sorted() is not cosmetic either: the fragment order fixes the row order of the test set,
    and therefore the order of y_true and of every saved prediction vector."""
    files = sorted(str(f) for f in Path(root).rglob("*.parquet"))
    if not files:
        raise RuntimeError(f"no .parquet files under {root}")
    return files


def _labels(col, num_classes=16):
    """astype(np.uint8) on a label of -1 gives 255 and on 300 gives 44 -- both are silent,
    and both survive every downstream assert because the counts still add up."""
    v = col.to_numpy(zero_copy_only=False)
    lo, hi = int(v.min()), int(v.max())
    if lo < 0 or hi >= num_classes:
        raise RuntimeError(f"labels out of range [{lo}, {hi}], expected 0..{num_classes-1}")
    return v.astype(np.uint8)


def load_clients(fl_root, feature_cols, n_clients, dtype=np.float16):
    """Returns X (N,66) fp16, y (N,) uint8, and spans[cid] = (lo, hi) contiguous row range.

    Contiguous spans are what make the training loop a slice + randperm instead of a
    gather over a client-id column."""
    dirs = sorted((fl_root / "train").glob("client_id=*"),
                  key=lambda p: int(p.name.split("=")[1]))
    if len(dirs) != n_clients:
        raise RuntimeError(f"expected {n_clients} client dirs, found {len(dirs)}")
    cols = list(feature_cols) + ["label"]
    xs, ys, spans, off = [], [], {}, 0
    for d in dirs:
        cid = int(d.name.split("=")[1])
        t = ds.dataset(parquet_files(d), format="parquet").to_table(columns=cols)
        n = t.num_rows
        a = np.empty((n, len(feature_cols)), dtype=dtype)
        for j, c in enumerate(feature_cols):
            a[:, j] = t.column(c).to_numpy(zero_copy_only=False).astype(dtype, copy=False)
        xs.append(a)
        ys.append(_labels(t.column("label")))
        spans[cid] = (off, off + n); off += n
        del t
    return np.concatenate(xs), np.concatenate(ys), spans


def load_test(test_root, feature_cols, scaler, dtype=np.float16):
    """test/ is raw: apply scaler.json here, and nowhere else."""
    t = ds.dataset(parquet_files(test_root), format="parquet").to_table(
        columns=list(feature_cols) + ["label"])
    n = t.num_rows
    X = np.empty((n, len(feature_cols)), dtype=dtype)
    for j, c in enumerate(feature_cols):
        v = t.column(c).to_numpy(zero_copy_only=False).astype(np.float64)
        v = np.nan_to_num(v, nan=0.0, posinf=0.0, neginf=0.0)
        s = scaler[c]
        X[:, j] = ((v - s["mean"]) / s["std_used"]).astype(dtype)
    y = _labels(t.column("label"))
    return X, y


def assert_fp16_safe(X, name):
    """knowledge/dataset.md measured max|x| = 570.44 on both splits, two orders below the
    fp16 ceiling. Assert it rather than inherit the assumption."""
    m = float(np.abs(X).max())
    if not np.isfinite(m) or m >= 65504:
        raise RuntimeError(f"{name}: max|x| = {m} is not fp16-safe")
    return m


def cache_ok(cache, want, n_clients, n_features=66):
    """Is the prepack cache complete, current and self-consistent?

    A matching manifest is a claim, not evidence. Running the notebook's own hit branch with
    a matching manifest and no train_X printed "cache reusable"; the worker then died opening
    a file that had never been written. Headers are read through mmap, so this costs a few
    stat calls and no data."""
    cache = Path(cache)
    mf = cache / "manifest.json"
    if not mf.is_file():
        return False
    try:
        if json.loads(mf.read_text()) != want:
            return False
        for f in CACHE_FILES:
            if not (cache / f).is_file() or (cache / f).stat().st_size == 0:
                return False
        spans = {int(k): tuple(v)
                 for k, v in json.load(open(cache / "spans.json")).items()}
        X = np.load(cache / "train_X.f16.npy", mmap_mode="r")
        Y = np.load(cache / "train_y.u8.npy", mmap_mode="r")
        TX = np.load(cache / "test_X.f16.npy", mmap_mode="r")
        TY = np.load(cache / "test_y.u8.npy", mmap_mode="r")
    except Exception as e:
        print("[cache] unreadable:", e)
        return False
    n = sum(hi - lo for lo, hi in spans.values())
    r = sorted(spans.values())
    return bool(
        len(spans) == n_clients
        and X.dtype == np.float16 and Y.dtype == np.uint8
        and TX.dtype == np.float16 and TY.dtype == np.uint8
        and X.shape == (n, n_features) and Y.shape == (n,)
        and TX.ndim == 2 and TX.shape[1] == n_features and len(TX) == len(TY)
        and r and r[0][0] == 0 and r[-1][1] == n
        and all(a[1] == b[0] for a, b in zip(r, r[1:])))


In [ ]:
%%writefile /kaggle/working/proj/perfedskd.py
"""PerFed-SKD (Singh, Rupchandani, Adhikari) — the self-knowledge-distillation personalized
FL loop: Eq. (2)-(3), Algorithm 1 (server) and Algorithm 2 (device). The paper's server-side
"Teacher Model trained on a predefined dataset" and any feature-extraction backbone are
deliberately absent; every model in the system is the same DAGSNet classifier
(proj/model.py) and the only knowledge transfer is the one the paper's name refers to —
from a device's OWN previous personalized model to its current one.

Round t, for EVERY client m (Algorithm 2 line 2 iterates over all M in parallel):

    V_m      <- omega_m^{t-1}                 the personalized model saved last round,
                                              FROZEN for the whole round (the teacher)
    omega_m  <- omega^{t-1}  if m in S_t       Algorithm 2 line 5  (selected: take global)
                omega_m^{t-1} otherwise        Algorithm 2 line 7  (keep own weights)

    one local epoch, per batch (x, y):
        z   = f(omega_m, x)                   student, train mode
        z_t = f(V_m, x)                       teacher, eval mode, no grad
        phi = CE(z, y) + lam * KL(p_t || p)                                 Eq. (2)
        omega_m <- omega_m - eta * grad phi                                 Eq. (3)

    upload omega_m^t  <=>  m in S_t           (only the selected devices report back)

Server (Algorithm 1):
    omega^t   = (1/|S_t|) sum_{m in S_t} omega_m^t                          line 10
    tau_t     = (1/M) sum_{m in M} a_m^t                                    line 11
    S_{t+1}   = { m : a_m^t < tau_t }                                       lines 4-6

a_m^t is client m's accuracy on the FIXED GLOBAL TEST SET, the same 10,761,343 rows for
every client (owner's decision, 2026-09-21): the paper says only "local model accuracy"
and a per-client test split does not exist in this partition.

---------------------------------------------------------------------------------------
What the paper leaves open, and what was fixed on 2026-09-21. Every one of these is a
deviation to publish with the numbers; the reasoning is in rebuild.md section 2.

  * `tau`. Algorithm 1 line 11 defines A as the MEAN OF THE CLIENTS' accuracies and line 4
    sets tau = A; the prose of section III-A instead says local accuracy is compared with
    "global aggregated model accuracy". Only the first is written as a formula, and it is
    the only one that selects a subset: on this data the aggregated model beats every
    personalized model on the global test (measured in the sibling rebuild: 0.79 against
    <= 0.70 f1_macro), so the prose reading would select all M clients in every round and
    the method would collapse into FedAvg + SKD. `tau = mean_m a_m` it is.
  * `a_m` is ACCURACY, as the paper writes it, not f1_macro. VeReMi is imbalanced 41:1, so
    accuracy is the weaker statistic here — but it is the paper's rule, and all 10 metrics
    are stored for every client and every round anyway.
  * WHO trains and WHO uploads. Algorithm 1 lines 8-9 loop over S alone; the prose of
    section IV and Algorithm 2 lines 2-7 say every device trains and the unselected ones
    simply keep their own parameters. The prose is followed: all M train, only S upload,
    and the sum in line 10 is over S (which is what its own 1/|S| divisor implies).
  * `L`, the "divergence function" of Eq. (2), is the KL divergence on softmax outputs at
    temperature 1, in the direction KL(teacher || student). `lam` = 1.0. The paper gives
    neither the family, the temperature nor the value.
  * the teacher runs in eval() mode (BatchNorm running statistics, dropout off) and under
    no_grad. A train()-mode teacher would answer with batch statistics and a random
    dropout mask, i.e. a different teacher for every batch.
  * round 1: S_1 = all M (no accuracies exist yet), and V_m = omega^0 for every m, so the
    distillation term starts at exactly 0 and grows as the student leaves its own
    initialization. From round 2 the teacher is a genuinely different model.
  * AdamW, weight decay 1e-4 (knowledge/architecture.md), re-created per client per round,
    so no optimizer state exists at a round boundary and a checkpoint is weights alone.
    The paper says only "Stochastic Gradient Descent" in Eq. (3) and names no rate.
  * the learning rate follows a per-ROUND cosine schedule (`lr_at`), constant within a
    round: the owner's single LR policy across these rebuilds.
"""
import contextlib
import math
import numpy as np
import torch
import torch.nn.functional as F


def amp(cfg):
    """fp16 autocast on CUDA; a no-op on CPU so the same code runs in the local
    simulation. Never bf16: the T4 is sm_75 and falls back to a slow emulation path."""
    if cfg.get("device", "cuda") == "cuda":
        return torch.autocast("cuda", dtype=torch.float16)
    return contextlib.nullcontext()


# --------------------------------------------------------------------- flat layout
# One flat float vector + one int vector per model. A DAGSNet state_dict has 192 entries;
# torch.multiprocessing gives each tensor its own shared-memory fd, so 100 clients a round
# would exhaust the process fd limit. Parameters come FIRST so vec[:n_params] is exactly
# the learnable block; BN running stats follow as buffers.
def layout(model):
    pnames = {n for n, _ in model.named_parameters()}
    sd = model.state_dict()
    fkeys = [k for k in sd if k in pnames]
    fkeys += [k for k in sd if k not in pnames and sd[k].is_floating_point()]
    ikeys = [k for k in sd if not sd[k].is_floating_point()]
    n_params = sum(sd[k].numel() for k in fkeys if k in pnames)
    return fkeys, ikeys, n_params


def flatten(model, fkeys, ikeys):
    sd = model.state_dict()
    fv = torch.cat([sd[k].reshape(-1).float() for k in fkeys])
    iv = torch.stack([sd[k].reshape(-1).long().squeeze() for k in ikeys]) if ikeys \
        else torch.zeros(0, dtype=torch.long)
    return fv, iv


def unflatten_into(model, fv, iv, fkeys, ikeys):
    sd = model.state_dict()
    o = 0
    for k in fkeys:
        t = sd[k]; n = t.numel()
        t.copy_(fv[o:o + n].view_as(t)); o += n           # copy_ keeps addresses -> CUDA graph valid
    for j, k in enumerate(ikeys):
        sd[k].copy_(iv[j].view_as(sd[k]))
    return model


# --------------------------------------------------------------------- learning rate
LR_SCHEDULES = ("constant", "cosine")


def lr_at(cfg, rnd):
    """Learning rate of round `rnd` (1-based), held constant within the round. A pure
    function of (cfg, rnd): a resumed session applies exactly the value the original
    session would have.

      constant : cfg['lr'] every round
      cosine   : lr_min + (lr - lr_min)/2 * (1 + cos(pi * (rnd-1) / (rounds-1)))
                 -- cfg['lr'] at round 1, cfg['lr_min'] at round cfg['rounds'].
    """
    sched = cfg.get("lr_schedule", "constant")
    if sched == "constant":
        return float(cfg["lr"])
    if sched == "cosine":
        T = int(cfg["rounds"])
        if not 1 <= rnd <= T:
            raise ValueError(f"round {rnd} outside 1..{T}: the cosine schedule is undefined")
        if T == 1:
            return float(cfg["lr"])
        lo, hi = float(cfg["lr_min"]), float(cfg["lr"])
        return lo + 0.5 * (hi - lo) * (1.0 + math.cos(math.pi * (rnd - 1) / (T - 1)))
    raise ValueError(f"lr_schedule {sched!r} not in {LR_SCHEDULES}")


# --------------------------------------------------------------------- the loss
# Order of the per-step values accumulated on the device (see `client_update`).
ACC_KEYS = ("loss", "ce", "kd", "gnorm")


def skd_loss(z, z_t, y, lam):
    """Eq. (2) for one batch, from fp32 logits.

        ce  = CE(z, y)                          f_m(omega_m^t), the empirical loss
        kd  = KL(p_t || p) = sum p_t (log p_t - log p)      L( x(V_m) || x(omega_m^t) )
        phi = ce + lam * kd

    The teacher's logits arrive already detached (they are produced under no_grad); the
    explicit .detach() here is what keeps that true if a caller ever forgets."""
    ce = F.cross_entropy(z, y)
    log_p = F.log_softmax(z, dim=1)
    log_pt = F.log_softmax(z_t, dim=1).detach()
    # kl_div(input=log q, target=log p, log_target=True) = sum p (log p - log q);
    # batchmean divides by the number of rows, so the scale does not follow the batch.
    kd = F.kl_div(log_p, log_pt, log_target=True, reduction="batchmean")
    return ce + lam * kd, (ce + lam * kd, ce, kd)


# --------------------------------------------------------------------- client update
def make_optimizer(Se, lr, cfg, fused):
    """AdamW over the student's parameters only. The teacher holds no gradient at all:
    it is loaded from the previous round's weights and never stepped."""
    return torch.optim.AdamW(list(Se.parameters()), lr=lr,
                             weight_decay=cfg["weight_decay"], fused=fused)


def client_update(Sc, Se, Tc, Te, opt, scaler, X, Y, lo, hi, cfg, gen):
    """`local_epochs` passes over rows [lo, hi) of the resident tensors for one client.

    Se is the eager student module that owns omega_m; Te the eager teacher module holding
    the frozen V_m. Sc/Tc are their compiled aliases (or the same objects when compile is
    off). The tail batch is a different shape and would recompile the CUDA graph once per
    client, so it runs on the eager modules: same weights, same math.

    Returns (acc, n_steps): `acc` is a device tensor of len(ACC_KEYS) + 2 sums over
    APPLIED steps (the last two entries are the skipped-step count and the count of
    applied steps whose gradient norm was not finite), read once by the caller.
    Dropout reads torch's default generator, which the worker re-seeds from
    (seed, round, client) before calling this; `gen` drives only the shuffles."""
    B, clip, dev, lam = cfg["batch"], cfg["clip"], X.device, float(cfg["lam"])
    params = list(Se.parameters())
    Se.train()
    Te.eval()                      # BatchNorm running stats, dropout off: a fixed teacher
    acc = torch.zeros(len(ACC_KEYS) + 2, device=dev)
    zeros = torch.zeros(len(ACC_KEYS), device=dev)
    n = 0
    for _ in range(cfg["local_epochs"]):
        perm = lo + torch.randperm(hi - lo, generator=gen, device=dev)
        for i in range(0, hi - lo, B):
            # Two forward graphs (teacher, student) and one backward run per step.
            # CUDA-graph trees decide which pool memory is dead from the iteration
            # boundary, so declare it explicitly instead of letting the teacher's forward
            # look like a new iteration and invalidate the student's.
            if X.is_cuda:
                torch.compiler.cudagraph_mark_step_begin()
            idx = perm[i:i + B]
            full = idx.numel() == B
            xb = X[idx].float()
            yb = Y[idx].long()
            with torch.no_grad(), amp(cfg):
                z_t = (Tc if full else Te)(xb)
            z_t = z_t.float()
            with amp(cfg):
                z = (Sc if full else Se)(xb)
            loss, parts = skd_loss(z.float(), z_t, yb, lam)    # loss in fp32
            scaler.scale(loss).backward()
            scaler.unscale_(opt)                               # grads now in true units
            gn = torch.nn.utils.clip_grad_norm_(params, clip)
            prev = scaler._scale.clone() if scaler.is_enabled() else None
            scaler.step(opt); scaler.update()
            opt.zero_grad(set_to_none=True)
            # A skipped step overflowed: its grad-norm is inf and its loss may be nan.
            # torch.where, not multiplication -- inf*0 is nan.
            applied = (scaler._scale >= prev) if prev is not None \
                else torch.ones((), dtype=torch.bool, device=dev)
            vals = torch.stack([p.detach() for p in parts] + [gn])
            acc[:len(ACC_KEYS)] += torch.where(applied, vals, zeros)
            acc[len(ACC_KEYS)] += (~applied).float()
            acc[len(ACC_KEYS) + 1] += ((~torch.isfinite(gn)) & applied).float()
            n += 1
    return acc, n


def expected_steps(n_k, cfg):
    return cfg["local_epochs"] * math.ceil(n_k / cfg["batch"])


# --------------------------------------------------------------------- server side
def aggregate(updates):
    """Algorithm 1 line 10: omega^{t+1} = (1/|S|) sum_{m in S} omega_m^{t+1}.

    `updates` holds ONLY the selected clients and must already be sorted by client id --
    float addition order decides the result, and it must not be set by a completion race.
    An empty S is not aggregated here: the driver keeps the previous global model, since
    a round in which nobody reported has produced no new global information."""
    if not updates:
        raise ValueError("aggregate() called with no selected client; the caller must "
                         "keep the previous global model instead")
    K = len(updates)
    acc = None
    for cid, fv, iv in updates:
        acc = fv / K if acc is None else acc.add_(fv, alpha=1.0 / K)
    # num_batches_tracked is an int counter, not an averageable quantity; with the default
    # BatchNorm momentum=0.1 it is unused at inference. Take the max so it stays monotone.
    ints = torch.stack([iv for _, _, iv in updates]).amax(dim=0) if updates[0][2].numel() \
        else updates[0][2]
    return acc, ints


SELECT_METRIC = "accuracy"          # the paper's a_m / A; see the module docstring


def threshold(acc_by_cid):
    """Algorithm 1 line 11: A <- (1/|M|) sum_{m in M} a_m, over ALL M clients -- including
    the ones that did not upload this round, which is what the sum's |M| divisor says."""
    v = np.asarray([acc_by_cid[c] for c in sorted(acc_by_cid)], dtype=np.float64)
    if v.size == 0:
        raise ValueError("threshold() needs at least one client accuracy")
    return float(v.mean())


def select_clients(acc_by_cid, tau):
    """Algorithm 1 lines 5-6: the devices whose accuracy is BELOW the threshold are the
    ones that receive the new global model. Strictly below, as `if a < tau` is written;
    ties keep their personalized weights."""
    return [c for c in sorted(acc_by_cid) if float(acc_by_cid[c]) < float(tau)]


In [ ]:
%%writefile /kaggle/working/proj/evaluate.py
"""Per-client evaluation of the personalized model f_s(w_s^k, x) on the fixed global test
set, plus the aggregated proxy f_r(w_bar_r, x).

The lightweight-FL NILM method keeps a personalized model on every device (only its proxy
is uploaded), so a round's evaluation is N + 1 independent full-test passes -- at 100
clients the dominant cost of the run.

Two exact speed-ups, both measured on 2xT4 in this repository's sibling projects:
  * BatchNorm folding: in eval mode BN is an affine map with constant coefficients, so it
    folds into the preceding Conv1d exactly (max|dlogit| 4.8e-07 measured), removing 31
    kernel launches per forward.
  * One folded TEMPLATE module per worker, compiled once with CUDA graphs. Client weights
    are folded and copied INTO the template with load_state_dict (in-place copy_), so
    parameter addresses never change and the captured graph stays valid for every client.
"""
import copy
import torch
import torch.nn as nn


@torch.no_grad()
def fold_bn(model):
    """Return an eval-mode copy with every BatchNorm folded into its preceding Conv1d.

        y = gamma*(conv(x) - mean)/sqrt(var + eps) + beta
          = conv'(x) + b'   with   w' = w*gamma/sqrt(var+eps),  b' = beta - gamma*mean/sqrt(var+eps)

    Exact for model.eval(); meaningless for model.train(). Every BatchNorm in DAGSNet sits
    inside a `cbr` block, i.e. Sequential(Conv1d, BN, ReLU), and the assertion at the end
    is what stops a future architecture change from silently leaving one unfolded."""
    m = copy.deepcopy(model).eval()
    for seq in m.modules():
        if not (isinstance(seq, nn.Sequential) and len(seq) >= 2
                and isinstance(seq[0], nn.Conv1d) and isinstance(seq[1], nn.BatchNorm1d)):
            continue
        conv, bn = seq[0], seq[1]
        inv = torch.rsqrt(bn.running_var + bn.eps)
        w = conv.weight * (bn.weight * inv).view(-1, 1, 1)
        b = bn.bias - bn.weight * bn.running_mean * inv
        if conv.bias is not None:
            b = b + conv.bias * bn.weight * inv
        new = nn.Conv1d(conv.in_channels, conv.out_channels, conv.kernel_size[0],
                        stride=conv.stride[0], padding=conv.padding[0], bias=True,
                        device=w.device, dtype=w.dtype)
        new.weight.copy_(w)
        new.bias.copy_(b)
        seq[0] = new
        seq[1] = nn.Identity()
    left = [n for n, mod in m.named_modules() if isinstance(mod, nn.BatchNorm1d)]
    assert not left, f"BatchNorm survived folding at {left}; the cbr pattern changed"
    for p in m.parameters():
        p.requires_grad_(False)
    return m.eval()


def load_folded(template, model):
    """Fold `model` and copy the result into `template` in place (same folded structure).
    strict=True: a key mismatch means the template was built from a different architecture."""
    template.load_state_dict(fold_bn(model).state_dict(), strict=True)
    return template


@torch.inference_mode()
def eval_model(compiled, eager, TX, TY, cfg, want_preds=False):
    """Confusion matrix of one folded model over the whole resident test set.

    Counts stay on the device: a per-batch .item() would sync ~650 times a pass, and the
    argmax of a row of NaN is 0 -- a perfectly ordinary class index -- so non-finite logits
    are counted explicitly instead of trusted. The tail batch runs eagerly (different shape
    would recompile the graph), same weights, same math."""
    C, EB, n = cfg["num_classes"], cfg["eval_batch"], TX.shape[0]
    dev = TX.device
    cm = torch.zeros(C * C, dtype=torch.long, device=dev)
    nonfin = torch.zeros((), dtype=torch.long, device=dev)
    preds = torch.empty(n, dtype=torch.uint8, device=dev) if want_preds else None
    ac = torch.autocast("cuda", dtype=torch.float16) if dev.type == "cuda" \
        else torch.autocast("cpu", enabled=False)
    for i in range(0, n, EB):
        j = min(i + EB, n)
        m = compiled if j - i == EB else eager
        # The previous batch's logits are still referenced by `z` when the next replay
        # starts; without an explicit step boundary CUDA-graph trees treat the call as
        # part of the same iteration and record a NEW graph node instead of replaying
        # (measured locally: 15k rows/s "compiled" vs 242k eager).
        if dev.type == "cuda":
            torch.compiler.cudagraph_mark_step_begin()
        with ac:
            z = m(TX[i:j].float())
        nonfin += (~torch.isfinite(z)).sum()
        p = z.argmax(1)
        cm += torch.bincount(TY[i:j].long() * C + p, minlength=C * C)
        if want_preds:
            preds[i:j] = p.to(torch.uint8)
    return cm.view(C, C), int(nonfin.item()), preds


In [ ]:
%%writefile /kaggle/working/proj/driver.py
"""One persistent worker per GPU, spawned once for the whole run.

Both GPUs hold the entire partition and the whole test set, so any client can train on
whichever GPU is free (longest-first dispatch) and evaluation splits the M personalized
models between the two GPUs. Each worker also holds a resident copy of EVERY client's
personalized weights omega_m (M x 1.58 MB) and of the server's aggregate omega, kept in
sync by the driver after each round, so a train task carries only a client id, its row
span and one boolean -- whether the server selected it this round.

That boolean is the whole of PerFed-SKD's device selection on the worker side:

    teacher V_m  <- CW[cid]          always: the client's own model from the last round
    student      <- GW if selected   Algorithm 2 line 5
                    CW[cid] if not   Algorithm 2 line 7

Aggregation covers the SELECTED clients only and is re-sorted by client id so float
addition order never depends on which worker finished first; every client re-seeds the
default generator from (seed, round, client) so its update does not depend on the
schedule either. Different clients are different models: they never form a process group.

Note on what "communication" means here. A real deployment transmits |S_t| models down
and |S_t| up; this simulator moves all M back to the driver because the driver owns the
client table. The saving the paper claims is reported as `comm_*` columns computed from
|S_t|, not measured from this process's queues.
"""
import json, math, shutil, time
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.multiprocessing as mp

from proj.model import build_model, N_PARAMS
from proj.perfedskd import (layout, flatten, unflatten_into, client_update, aggregate,
                            amp, lr_at, make_optimizer, skd_loss, threshold,
                            select_clients, ACC_KEYS, SELECT_METRIC)
from proj.evaluate import fold_bn, load_folded, eval_model
from proj.metrics import metrics_from_confusion, per_class_from_confusion, METRIC_KEYS
from proj import ckpt as C

# 395,024 fp32 parameters + 3,295 BatchNorm buffers is what one model costs on the wire.
MODEL_MIB = (N_PARAMS + 3295) * 4 / 2**20


def _resident(path, dev, chunk=1 << 22):
    """mmap -> GPU in chunks. A whole-array np.ascontiguousarray would materialise 5.7 GB
    of train features in host RAM per worker before the copy, and hands torch a read-only
    array. Chunking bounds the host side to `chunk` rows."""
    a = np.load(path, mmap_mode="r")
    t = torch.empty(tuple(a.shape), dtype=torch.from_numpy(np.array(a[:1])).dtype,
                    device=dev)
    for i in range(0, len(a), chunk):
        t[i:i + chunk] = torch.from_numpy(np.array(a[i:i + chunk]))
    return t


def _verdict(ref_z, got_z, ref_g, got_g, n_rows, label):
    """Decisive-row argmax agreement plus bounded logit/gradient deltas. Counts are
    integers: torch.mean on CUDA returns 0.99999994 for a perfect match."""
    dz = (got_z - ref_z).abs().max().item()
    gn = ref_g.norm().item()
    dg = (got_g - ref_g).norm().item() / (gn + 1e-12)
    flip_all = int((got_z.argmax(1) != ref_z.argmax(1)).sum())
    top2 = ref_z.topk(2, dim=1).values
    decisive = (top2[:, 0] - top2[:, 1]) > max(10 * dz, 1e-3)
    n_dec = int(decisive.sum())
    flip_dec = int((got_z.argmax(1) != ref_z.argmax(1))[decisive].sum())
    # `flip_dec == 0` over an EMPTY decisive set says nothing at all.
    if n_dec < n_rows // 10:
        raise RuntimeError(f"{label}: cannot certify, only {n_dec} of {n_rows} rows have "
                           f"a margin above {max(10 * dz, 1e-3):.2e}")
    if flip_dec or not math.isfinite(dz) or not math.isfinite(dg) or dz > 5e-2 or dg > 5e-2:
        raise RuntimeError(f"{label}: mismatch dlogit={dz} dgrad_rel={dg} "
                           f"flips {flip_dec}/{n_dec} decisive, {flip_all} of all")
    return f"max|dlogit|={dz:.2e} rel|dgrad|={dg:.2e} flips {flip_all}/{n_rows} ({flip_dec}/{n_dec} decisive)"


def _compile_train(Se, Ve, cfg, dev, xb, yb, rank=0):
    """reduce-overhead captures forward+backward into a CUDA graph. torch.compile is lazy,
    so a try around the call catches nothing -- run the production SKD step and compare
    against eager from the SAME state, with Dropout off on the student (Inductor
    functionalises RNG, so the two masks can never coincide; the teacher is in eval() so
    its dropout is already off). Warm-up captures the graphs at the production p, so
    restoring p reuses those entries.

    Both models are gated: the teacher's logits enter the loss, so a wrong teacher graph
    is a wrong objective, not merely a slow one."""
    if not cfg["compile"]:
        return Se, Ve
    drops = [m for m in Se.modules() if isinstance(m, nn.Dropout)]
    keep = [m.p for m in drops]
    snap = [{k: v.detach().clone() for k, v in m.state_dict().items()} for m in (Se, Ve)]
    rng = torch.get_rng_state()
    crng = torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None

    def restore():
        with torch.no_grad():
            for m, s in zip((Se, Ve), snap):
                sd = m.state_dict()
                for k, v in s.items():
                    sd[k].copy_(v)                  # copy_ keeps addresses -> graph stays valid
        torch.set_rng_state(rng)
        if crng is not None: torch.cuda.set_rng_state_all(crng)
        Se.zero_grad(set_to_none=True)
        Se.train(); Ve.eval()

    def probe(Sm, Vm):
        """The production step: teacher forward under no_grad, student forward under
        autocast, Eq. (2) in fp32, one backward. Returns the two logit blocks and the
        student's flat gradient."""
        restore()
        if xb.is_cuda:
            torch.compiler.cudagraph_mark_step_begin()
        with torch.no_grad(), amp(cfg):
            z_t = Vm(xb)
        z_t = z_t.float()
        with amp(cfg):
            z = Sm(xb)
        z = z.float()
        loss, _ = skd_loss(z, z_t, yb, float(cfg["lam"]))
        loss.backward()
        g = torch.cat([p.grad.reshape(-1).float().clone() for p in Se.parameters()])
        Se.zero_grad(set_to_none=True)
        return z.clone(), g, z_t.clone()

    try:
        Sc = torch.compile(Se, mode="reduce-overhead")     # CUDA graphs: the 2.9x on T4
        Vc = torch.compile(Ve, mode="reduce-overhead")
        for _ in range(3):                                  # warm up + capture at production p
            probe(Sc, Vc)
        for m in drops: m.p = 0.0
        try:
            ref = probe(Se, Ve)
            got = probe(Sc, Vc)
        finally:
            for m, p_ in zip(drops, keep): m.p = p_
        restore()
        n = xb.shape[0]
        z0 = torch.zeros(1, device=dev)
        s1 = _verdict(ref[0], got[0], ref[1], got[1], n, "student")
        s2 = _verdict(ref[2], got[2], z0, z0, n, "teacher")
        print(f"[rank{rank}] compile OK | student {s1} | teacher {s2}", flush=True)
        return Sc, Vc
    except Exception as e:                        # sm_75 Triton is the documented risk
        for m, p_ in zip(drops, keep): m.p = p_   # never leave the model with dropout off
        restore()
        print(f"[rank{rank}] compile DISABLED -> eager: {e}", flush=True)
        return Se, Ve


def _compile_eval(Te, cfg, dev, xt, rank=0):
    """The folded eval template, compiled at the fixed eval batch. Certified against the
    eager folded template on real test rows: fp16 cannot be bit-equal, so the criterion is
    the decisive-row argmax rule with a delta ceiling."""
    if not cfg["compile"]:
        return Te
    try:
        Tc = torch.compile(Te, mode="reduce-overhead")
        with torch.inference_mode():
            for _ in range(3):
                with amp(cfg):
                    Tc(xt).float()
            with amp(cfg):
                ref = Te(xt).float().clone()
                got = Tc(xt).float().clone()
        z = torch.zeros(1, device=dev)
        s = _verdict(ref, got, z, z, xt.shape[0], "eval")
        print(f"[rank{rank}] eval compile OK | {s}", flush=True)
        return Tc
    except Exception as e:
        print(f"[rank{rank}] eval compile DISABLED -> eager: {e}", flush=True)
        return Te


def worker(rank, cfg, task_q, res_q):
    """Wrapper: a worker that dies silently leaves the parent with only an exit code, and
    the real error is always in the CHILD traceback, not the spawn wrapper."""
    try:
        _worker(rank, cfg, task_q, res_q)
    except Exception:
        import traceback
        res_q.put(("error", rank, traceback.format_exc()))
        raise


def _worker(rank, cfg, task_q, res_q):
    cuda = cfg.get("device", "cuda") == "cuda"
    dev = torch.device(f"cuda:{rank}" if cuda else "cpu")
    if cuda:
        torch.cuda.set_device(dev)
        torch.backends.cudnn.benchmark = True
    # The student, the teacher and the folded eval template share ONE code object
    # (DAGSNet.forward), and Dynamo caches per code object: train/eval mode, dropout on/off
    # for the gate, no_grad vs grad, and two batch shapes can pass the default recompile
    # limit of 8. Past the limit Dynamo runs the new variant EAGERLY without raising, so
    # the eval template would silently lose CUDA graphs (measured in a sibling project:
    # "compiled" eval 0.7x eager).
    for name in ("recompile_limit", "cache_size_limit"):
        if hasattr(torch._dynamo.config, name):
            setattr(torch._dynamo.config, name, 64)
    torch.manual_seed(cfg["seed"] + rank)
    cache = Path(cfg["cache"])
    X = _resident(cache / "train_X.f16.npy", dev)
    Y = _resident(cache / "train_y.u8.npy", dev)
    TX = _resident(cache / "test_X.f16.npy", dev)
    TY = _resident(cache / "test_y.u8.npy", dev)

    Se = build_model(cfg).to(dev).train()                # student omega_m (per task)
    Ve = build_model(cfg).to(dev).eval()                 # frozen teacher V_m (per task)
    for p in Ve.parameters():
        p.requires_grad_(False)                          # no graph is ever built for it
    fk, ik, nP = layout(Se)
    assert nP == N_PARAMS, nP
    B = cfg["batch"]
    # Probe on real rows: random N(0,1) has none of the heavy tails of the z-scored
    # features, and a kernel that is wrong only at large magnitude would pass on noise.
    Sc, Vc = _compile_train(Se, Ve, cfg, dev, X[:B].float(), Y[:B].long(), rank)
    Te = fold_bn(build_model(cfg).to(dev))           # folded STRUCTURE; weights per model
    Tc = _compile_eval(Te, cfg, dev, TX[:cfg["eval_batch"]].float(), rank)
    scaler_probe = torch.amp.GradScaler("cuda", enabled=cuda)
    if cuda:
        scaler_probe.scale(torch.zeros(1, device=dev))  # force _scale to exist
        assert scaler_probe._scale is not None, "GradScaler._scale gone; skips would read as 0"
    res_q.put(("ready", rank, "eager" if Sc is Se else "compiled",
               "eager" if Tc is Te else "compiled"))

    CW = CI = GW = GI = None
    while True:
        task = task_q.get()
        kind = task[0]
        if kind == "stop":
            return
        if kind == "backend":
            # Both ranks gate independently, so one can compile and the other fall back.
            # A round whose clients were trained on two different backends is not a round
            # anyone can reproduce; the driver forces the lower common denominator.
            if task[1] == "eager": Sc, Vc = Se, Ve
            if task[2] == "eager": Tc = Te
            res_q.put(("backend_ok", rank, "eager" if Sc is Se else "compiled",
                       "eager" if Tc is Te else "compiled"))
            continue
        # Payloads cross the process boundary as numpy arrays: a torch tensor on a
        # multiprocessing queue is shared through /dev/shm, which a container may cap at
        # 64 MB, and the resident client table is 160 MB at 100 clients.
        if kind == "init":
            CW, CI = torch.from_numpy(task[1]).to(dev), torch.from_numpy(task[2]).to(dev)
            GW, GI = torch.from_numpy(task[3]).to(dev), torch.from_numpy(task[4]).to(dev)
            res_q.put(("init_ok", rank))
            continue
        if kind == "set_clients":
            cids = task[1]
            fvs, ivs = torch.from_numpy(task[2]).to(dev), torch.from_numpy(task[3]).to(dev)
            for j, c in enumerate(cids):
                CW[c].copy_(fvs[j]); CI[c].copy_(ivs[j])
            res_q.put(("set_ok", rank))
            continue
        if kind == "set_global":
            GW.copy_(torch.from_numpy(task[1]).to(dev)); GI.copy_(torch.from_numpy(task[2]).to(dev))
            res_q.put(("set_ok", rank))
            continue
        if kind == "eval":
            cids, want_preds, with_global = task[1], task[2], task[3]
            t0 = time.monotonic()
            if cuda: torch.cuda.reset_peak_memory_stats(dev)
            cms, nfs, preds = [], [], []
            for c in cids:
                unflatten_into(Se, CW[c], CI[c], fk, ik)
                load_folded(Te, Se)
                cm, nf, p = eval_model(Tc, Te, TX, TY, cfg, want_preds)
                cms.append(cm.cpu()); nfs.append(nf)
                if want_preds: preds.append(p.cpu())
            g = None
            if with_global:                                 # the server's aggregate omega
                unflatten_into(Se, GW, GI, fk, ik)
                load_folded(Te, Se)
                cm, nf, p = eval_model(Tc, Te, TX, TY, cfg, want_preds)
                g = (cm.cpu().numpy(), nf, p.cpu().numpy() if want_preds else None)
            res_q.put(("eval", rank, list(cids),
                       torch.stack(cms).numpy() if cms else None, nfs,
                       torch.stack(preds).numpy() if preds else None, g,
                       (torch.cuda.max_memory_allocated(dev) / 2**30) if cuda else 0.0,
                       time.monotonic() - t0))
            continue
        if kind == "train":
            cid, lo, hi, rnd, selected = task[1], task[2], task[3], task[4], task[5]
            t0 = time.monotonic()
            if cuda: torch.cuda.reset_peak_memory_stats(dev)
            # V_m is ALWAYS the client's own model from the previous round (Algorithm 2
            # line 12 of the round before). The student starts from the global model only
            # if the server selected this device; otherwise it continues from V_m, and the
            # distillation term is then an anchor to where the client itself left off.
            unflatten_into(Ve, CW[cid], CI[cid], fk, ik)
            if selected:
                unflatten_into(Se, GW, GI, fk, ik)            # Algorithm 2 line 5
            else:
                unflatten_into(Se, CW[cid], CI[cid], fk, ik)  # Algorithm 2 line 7
            # AdamW re-created per client per round (owner's decision): a selected client
            # receives fresh weights from the server, and carrying moments across that
            # discontinuity would apply the previous model's curvature to a new one.
            # fused=True collapses the step into one multi-tensor kernel.
            lr = lr_at(cfg, rnd)
            opt = make_optimizer(Se, lr, cfg, fused=cuda)
            scaler = torch.amp.GradScaler("cuda", enabled=cuda)
            if cuda:
                scaler.scale(torch.zeros(1, device=dev))
            # Every stochastic input to this client derives from (seed, round, client):
            # the default generator drives Dropout, `g` drives the shuffles.
            s = cfg["seed"] * 1_000_003 + rnd * 10_007 + cid
            torch.manual_seed(s)
            g = torch.Generator(device=dev); g.manual_seed(s)
            acc, n = client_update(Sc, Se, Vc, Ve, opt, scaler, X, Y, lo, hi, cfg, g)
            sv, si = flatten(Se, fk, ik)
            a = acc.cpu().tolist()
            sk = int(a[len(ACC_KEYS)]); ap = n - sk                # NOT max(1, .): 0 must stay 0
            d_ = max(1, ap)
            st = {"round": rnd, "cid": cid, "n_k": hi - lo, "rank": rank, "seed": s,
                  "selected": bool(selected),
                  "lr": lr, "steps": n, "applied": ap, "skipped": sk,
                  "nonfinite": int(a[len(ACC_KEYS) + 1]),
                  "sec": time.monotonic() - t0,
                  "vram_gb": (torch.cuda.max_memory_allocated(dev) / 2**30 if cuda else 0.0)}
            st.update({k: a[j] / d_ for j, k in enumerate(ACC_KEYS)})   # means over applied steps
            res_q.put(("train", cid, hi - lo, sv.cpu().numpy(), si.cpu().numpy(), st))


def check_updates(rnd, results, stats, n_clients, max_skips=None):
    """Every reason a round must not be aggregated, in one pure function so it can be
    tested without two GPUs and a spawned worker.

    Skipped steps are NOT a failure: each client starts a fresh GradScaler at 2**16 and
    spends a few steps calibrating. What must be rejected is a client that applied no
    step, one that skipped far more than calibration explains, one whose APPLIED steps
    carried a non-finite gradient, and non-finite weights."""
    if sorted(stats) != list(range(n_clients)):
        raise RuntimeError(f"round {rnd}: reported {sorted(stats)}, expected 0..{n_clients - 1}")
    bad = [cid for cid, _, sv, _, _ in results if not np.isfinite(sv).all()]
    if bad:
        raise RuntimeError(f"round {rnd}: non-finite weights from clients {bad}")
    for c in sorted(stats):
        st = stats[c]
        if st["applied"] + st["skipped"] != st["steps"]:
            raise RuntimeError(f"round {rnd}: client {c} applied+skipped != steps")
    dead = [c for c in sorted(stats) if stats[c]["applied"] == 0]
    if dead:
        raise RuntimeError(f"round {rnd}: clients {dead} applied zero steps; "
                           "they would contribute unchanged weights")
    diverged = [c for c in sorted(stats) if stats[c]["nonfinite"]]
    if diverged:
        raise RuntimeError(f"round {rnd}: clients {diverged} APPLIED a step whose "
                           "gradient was not finite")
    if max_skips is not None:
        over = [c for c in sorted(stats) if stats[c]["skipped"] > max_skips]
        if over:
            raise RuntimeError(
                f"round {rnd}: clients {over} skipped more steps than the warm-up "
                f"budget ({max_skips}): "
                + ", ".join(f"{c}={stats[c]['skipped']}" for c in over))


def _collect(res_q, procs, n, timeout=7200):
    """A worker killed by the OS puts nothing on the queue. Poll in short slices and check
    liveness between them, or an OOM kill becomes a multi-hour hang."""
    out, deadline = [], time.time() + timeout
    while len(out) < n:
        try:
            msg = res_q.get(timeout=2.0)
            if msg[0] == "error":
                raise RuntimeError(f"worker {msg[1]} raised:\n{msg[2]}")
            out.append(msg)
        except RuntimeError:
            raise
        except Exception:
            for p in procs:
                if not p.is_alive() and p.exitcode not in (0, None):
                    raise RuntimeError(f"worker {p.pid} died, exitcode {p.exitcode} "
                                       f"(negative = signal; -9 is the OOM killer)")
            if time.time() > deadline:
                raise RuntimeError(f"timed out waiting for {n - len(out)} results")
    return out


def _shutdown(procs, task_qs):
    for q in task_qs:
        try: q.put(("stop",))
        except Exception: pass
    for p in procs:
        p.join(timeout=60)
        if p.is_alive():
            p.terminate(); p.join(timeout=10)


def _broadcast(task_qs, res_q, procs, msg):
    for q in task_qs: q.put(msg)
    return _collect(res_q, procs, len(task_qs))


def run(cfg, spans, class_names, wandb_run=None, t_origin=None):
    """t_origin is a time.monotonic() reading from when the SESSION started, not from when
    this call did. Worker spawn, the resident copy and compilation are minutes the 12 h cap
    charges for, and a deadline that started here would happily begin a round the session
    cannot finish."""
    # cfg['select_metric'] exists so ckpt.FINGERPRINT_KEYS can see the selection rule;
    # the rule itself is the module constant. If the two ever disagree, the fingerprint
    # would describe a run that did not happen.
    if cfg.get("select_metric") != SELECT_METRIC:
        raise SystemExit(f"cfg['select_metric'] = {cfg.get('select_metric')!r} but "
                         f"proj.perfedskd selects on {SELECT_METRIC!r}")
    t_start = t_origin if t_origin is not None else time.monotonic()
    mp.set_start_method("spawn", force=True)
    ctx = mp.get_context("spawn")
    task_qs = [ctx.Queue() for _ in range(cfg["world_size"])]
    res_q = ctx.Queue()
    procs = [ctx.Process(target=worker, args=(r, cfg, task_qs[r], res_q), daemon=True)
             for r in range(cfg["world_size"])]
    try:
        for p in procs: p.start()
        ready = _collect(res_q, procs, cfg["world_size"], timeout=3600)
        bt = {m[1]: m[2] for m in ready}; be = {m[1]: m[3] for m in ready}
        if len(set(bt.values())) > 1 or len(set(be.values())) > 1:
            print(f"[driver] ranks disagree on backend train={bt} eval={be}; forcing eager",
                  flush=True)
            force = ("backend", "eager" if len(set(bt.values())) > 1 else "keep",
                     "eager" if len(set(be.values())) > 1 else "keep")
            acks = _broadcast(task_qs, res_q, procs, force)
            bt = {m[1]: m[2] for m in acks}; be = {m[1]: m[3] for m in acks}
        cfg["backend"] = sorted(set(bt.values()))[0]
        cfg["backend_eval"] = sorted(set(be.values()))[0]
        startup = time.monotonic() - t_start
        print(f"[driver] {cfg['world_size']} workers ready: train {cfg['backend']}, "
              f"eval {cfg['backend_eval']} ({startup:.0f}s into the session)", flush=True)
        cfg["startup_seconds"] = startup
        # Push the effective backend somewhere READABLE WHILE THE RUN IS ALIVE: a running
        # Kaggle kernel's stdout cannot be downloaded.
        if wandb_run is not None:
            try:
                wandb_run.config.update({"backend": cfg["backend"],
                                         "backend_eval": cfg["backend_eval"],
                                         "startup_seconds": round(startup, 1)},
                                        allow_val_change=True)
                wandb_run.summary["backend"] = cfg["backend"]
                wandb_run.summary["backend_eval"] = cfg["backend_eval"]
            except Exception as e:
                print(f"[driver] could not publish backend to W&B: {e}", flush=True)
        return _rounds(cfg, spans, class_names, wandb_run, t_start, procs, task_qs, res_q)
    finally:
        # Without this a driver-side exception leaves two processes holding both GPUs, and
        # the next cell in the notebook fails with a CUDA OOM that names nothing.
        _shutdown(procs, task_qs)


def _stats(values):
    v = np.asarray(values, dtype=np.float64)
    return float(v.mean()), float(v.std()), float(v.min()), float(v.max())


def _rounds(cfg, spans, class_names, wandb_run, t_start, procs, task_qs, res_q):
    d = C.run_dir(cfg["run_name"])
    N, W = cfg["n_clients"], cfg["world_size"]
    torch.manual_seed(cfg["seed"])                 # seed BEFORE building: omega^0 seeded
    M0 = build_model(cfg)
    fk, ik, nP = layout(M0)
    f0v, f0i = flatten(M0, fk, ik)
    # Every model starts from the SAME seeded initialization: the M personalized models
    # AND the server's aggregate are copies of omega^0, so round 1's teacher V_m = omega^0
    # and the distillation term starts at exactly 0.
    CW = f0v.unsqueeze(0).repeat(N, 1).contiguous()
    CI = f0i.unsqueeze(0).repeat(N, 1).contiguous()
    GW, GI = f0v.clone(), f0i.clone()
    # Round 1 has no accuracies to threshold, so every device is selected (Algorithm 1
    # line 2 initializes omega^0 and line 5 has nothing to compare against yet).
    selected = list(range(N))

    start = 1
    last = C.resolve_resume(cfg["run_name"], cfg)
    if last is not None:
        G, Ms, ck = C.load_weights(d / "weights" / f"round_{last:03d}.pt",
                                   build_model, N_PARAMS)
        GW, GI = flatten(G, fk, ik)
        for c, m in Ms.items():
            CW[c], CI[c] = flatten(m, fk, ik)
        # The subset the interrupted session had already chosen for the next round. It is
        # in the checkpoint, not recomputed: recomputing would make a resumed run depend
        # on floating-point details of a comparison the original run had already made.
        selected = [int(c) for c in ck["selected_next"]]
        start = last + 1
        print(f"[driver] resumed at round {start} with |S| = {len(selected)} selected devices")
    elif cfg.get("require_resume"):
        raise SystemExit("require_resume set and no checkpoint found")
    if start > cfg["rounds"]:
        print(f"[driver] nothing to do: {last} rounds already complete")
        return []
    _broadcast(task_qs, res_q, procs, ("init", CW.numpy(), CI.numpy(), GW.numpy(), GI.numpy()))

    n_test = cfg["n_test"]
    preds_rounds = set(cfg.get("preds_rounds", [cfg["rounds"]]))
    hist = []
    reserve = cfg.get("finalize_reserve_seconds", 600)
    elapsed = time.monotonic() - t_start
    if elapsed + reserve >= cfg["max_seconds"]:
        print(f"[driver] no round started: {elapsed/3600:.2f} h of the "
              f"{cfg['max_seconds']/3600:.2f} h budget is already gone", flush=True)
        return hist

    for rnd in range(start, cfg["rounds"] + 1):
        t0 = time.monotonic()
        sel_set = set(selected)
        # Algorithm 2 line 2: every device trains, selected or not. Longest-first bounds
        # the idle tail: sending the biggest client last strands a GPU.
        order = sorted(range(N), key=lambda c: spans[c][1] - spans[c][0], reverse=True)
        pending, nxt, results = {}, 0, []
        for r in range(W):                                  # prime both GPUs
            if nxt < len(order):
                c = order[nxt]; nxt += 1
                task_qs[r].put(("train", c, *spans[c], rnd, c in sel_set)); pending[r] = c
        while len(results) < len(order):
            msg = _collect(res_q, procs, 1)[0]
            assert msg[0] == "train", msg[0]
            results.append(msg[1:])
            r = next(k for k, v in pending.items() if v == msg[1])
            if nxt < len(order):
                c = order[nxt]; nxt += 1
                task_qs[r].put(("train", c, *spans[c], rnd, c in sel_set)); pending[r] = c
            else:
                pending.pop(r)
        t_train = time.monotonic() - t0

        results.sort(key=lambda t: t[0])                    # NOT completion order
        stats = {cid: s for cid, _, _, _, s in results}
        check_updates(rnd, results, stats, N, cfg.get("max_skips_per_client"))
        # Algorithm 1 line 10: the aggregate is over the SELECTED devices only. An empty S
        # cannot happen once accuracies differ (tau is their mean), but if it ever did the
        # round produced no uplink and the previous global model stands.
        ups = [(cid, torch.from_numpy(sv), torch.from_numpy(si))
               for cid, _, sv, si, _ in results if cid in sel_set]
        if ups:
            GW, GI = aggregate(ups)
        else:
            print(f"[driver] round {rnd}: S is empty, keeping the previous global model",
                  flush=True)
        cids = [t[0] for t in results]
        svs = np.stack([t[2] for t in results]); sis = np.stack([t[3] for t in results])
        for j, c in enumerate(cids):
            CW[c].copy_(torch.from_numpy(svs[j])); CI[c].copy_(torch.from_numpy(sis[j]))
        _broadcast(task_qs, res_q, procs, ("set_clients", cids, svs, sis))
        _broadcast(task_qs, res_q, procs, ("set_global", GW.numpy(), GI.numpy()))

        # ---- evaluate every personalized model omega_m and the server's aggregate omega
        # on the full test set. Client c is always evaluated on worker c % W (the two
        # workers are separate processes whose cuDNN algorithm choice can differ on a few
        # fp16 rows; a fixed assignment keeps each client's series on one GPU). The
        # aggregate goes to the last worker, which holds the fewer clients when N is odd.
        want_preds = rnd in preds_rounds
        eval_split = [[c for c in range(N) if c % W == r] for r in range(W)]
        t1 = time.monotonic()
        for r in range(W):
            task_qs[r].put(("eval", eval_split[r], want_preds, r == W - 1))
        ev = _collect(res_q, procs, W)
        t_eval = time.monotonic() - t1
        fresh, g = {}, None
        preds = np.empty((N, n_test), dtype=np.uint8) if want_preds else None
        nf_total = 0
        for e in ev:
            for j, c in enumerate(e[2]):
                fresh[c] = e[3][j]; nf_total += e[4][j]
                if want_preds: preds[c] = e[5][j]
            if e[6] is not None:
                g = e[6]; nf_total += g[1]
        assert sorted(fresh) == list(range(N)) and g is not None, f"evaluated {sorted(fresh)}"
        if nf_total:
            raise RuntimeError(f"round {rnd}: {nf_total} non-finite test logits; argmax "
                               "would have turned them into ordinary class labels")
        cm = np.stack([fresh[c] for c in range(N)])
        gcm = g[0]
        assert (cm.sum(axis=(1, 2)) == n_test).all() and gcm.sum() == n_test, \
            "a confusion matrix misses test rows"
        per_client = []
        for c in range(N):
            m = metrics_from_confusion(cm[c])
            per_client.append({"cid": c, **m,
                               "per_class": per_class_from_confusion(cm[c], class_names)})
        gm = metrics_from_confusion(gcm)

        # ---- Algorithm 1 lines 11 and 4-6: the threshold, then next round's subset.
        acc_by_cid = {c: per_client[c][SELECT_METRIC] for c in range(N)}
        tau = threshold(acc_by_cid)
        selected_next = select_clients(acc_by_cid, tau)

        row = {"round": rnd, "evaluated": N, "lr": lr_at(cfg, rnd)}
        for k in METRIC_KEYS:                                # mean over ALL N clients
            mean, std, lo, hi = _stats([pc[k] for pc in per_client])
            row[k] = mean
            row[f"{k}_std"], row[f"{k}_min"], row[f"{k}_max"] = std, lo, hi
        for k in METRIC_KEYS:                                # the server's aggregate
            row[f"global_{k}"] = gm[k]
        # *_client_mean is the unweighted mean ACROSS CLIENTS of each client's mean over
        # its applied steps -- not the mean over training samples.
        for k in ACC_KEYS:
            row[f"{k}_client_mean"] = float(np.mean([s[k] for s in stats.values()]))
        # What the protocol would actually have transmitted this round: |S_t| models down
        # at the start and |S_t| up at the end. FedAvg's cost is 2 * N * MODEL_MIB.
        row.update({
            "n_selected": len(selected), "tau": tau, "n_selected_next": len(selected_next),
            "select_metric": SELECT_METRIC,
            "comm_down_mib": len(selected) * MODEL_MIB,
            "comm_up_mib": len(selected) * MODEL_MIB,
            "comm_saving": 1.0 - len(selected) / N,
            "steps": int(sum(s["steps"] for s in stats.values())),
            "skipped": int(sum(s["skipped"] for s in stats.values())),
            "train_sec": t_train, "eval_sec": t_eval,
            "vram_train_gb": max(s["vram_gb"] for s in stats.values()),
            "vram_eval_gb": max(e[7] for e in ev),
            "backend": cfg.get("backend", "?"), "seconds": 0.0})
        mean_metrics = {k: row[k] for k in METRIC_KEYS}

        # ---- commit. Marker absolutely last.
        unflatten_into(M0, GW, GI, fk, ik)
        global_sd = C.cpu_sd(M0)
        clients_sd = {}
        for c in range(N):
            unflatten_into(M0, CW[c], CI[c], fk, ik)
            clients_sd[c] = C.cpu_sd(M0)
        C.save_round_weights(global_sd, clients_sd, rnd, cfg, mean_metrics, gm, d,
                             selected, selected_next, tau)
        C.atomic_np_save(d / "confusion" / f"round_{rnd:03d}.npy", cm)
        C.atomic_np_save(d / "confusion" / f"global_{rnd:03d}.npy", gcm)
        if want_preds:
            C.atomic_np_save(d / "preds" / f"round_{rnd:03d}.u8.npy", preds)
            C.atomic_np_save(d / "preds" / f"global_{rnd:03d}.u8.npy", g[2])
        (d / "logs" / f"round_{rnd:03d}.json").write_text(
            json.dumps({"clients": [stats[c] for c in sorted(stats)],
                        "selected": [int(c) for c in selected],
                        "selected_next": [int(c) for c in selected_next],
                        "tau": tau}, indent=1))
        # `seconds` BEFORE the W&B call and the JSON: the budget below compares absolute
        # session elapsed, so the commit tail lands in the next round's elapsed and the
        # finalize reserve covers the last one.
        row["seconds"] = time.monotonic() - t0
        if wandb_run is not None:
            # W&B is a monitor, never a dependency: in a resumed session the service has
            # died at startup before and every later log call went nowhere. The round's
            # artifacts below are the record; a W&B failure must not touch them.
            try:
                wandb_run.log({k: v for k, v in row.items()
                               if k not in ("round", "backend", "select_metric")}, step=rnd)
            except Exception as e:
                print(f"[driver] W&B log failed at round {rnd}: {e}", flush=True)
        (d / "metrics" / f"round_{rnd:03d}.json").write_text(json.dumps(
            {**row, "selected": [int(c) for c in selected],
             "selected_next": [int(c) for c in selected_next],
             "clients": per_client,
             "global": {**gm, "per_class": per_class_from_confusion(gcm, class_names)}},
            indent=1))
        C.append_history(d, row, [{"round": rnd, **{k: v for k, v in pc.items()
                                                    if k != "per_class"}}
                                  for pc in per_client])
        C.mark_complete(d, rnd)
        hist.append(row)
        print(f"[r{rnd:03d}] f1_macro mean={row['f1_macro']:.6f} "
              f"std={row['f1_macro_std']:.4f} min={row['f1_macro_min']:.4f} "
              f"acc={row['accuracy']:.6f} | global f1={row['global_f1_macro']:.6f} "
              f"acc={row['global_accuracy']:.6f} "
              f"| |S|={len(selected)}->{len(selected_next)} tau={tau:.6f} "
              f"| lr={row['lr']:.2e} ce={row['ce_client_mean']:.4f} "
              f"kd={row['kd_client_mean']:.4f} "
              f"skip={row['skipped']}/{row['steps']} "
              f"train={t_train:.0f}s eval={t_eval:.0f}s vram={row['vram_train_gb']:.2f}G "
              f"{row['seconds']:.1f}s | session {(time.monotonic()-t_start)/3600:.2f}h",
              flush=True)
        selected = selected_next

        worst = max(h["seconds"] for h in hist)
        if (time.monotonic() - t_start) + worst * 1.15 + reserve > cfg["max_seconds"]:
            print(f"[driver] stopping after round {rnd}: the next round plus a "
                  f"{reserve/60:.0f} min finalize reserve would exceed the session budget "
                  f"({cfg['max_seconds']/3600:.2f} h)", flush=True)
            break

    return hist


def write_manifest(cfg, class_names, spans, y_true_src=None, extra=None):
    """Everything needed to say what these numbers are, written once, next to them.

    Also the SECOND data gate: `content_id` is computed after the decode from the row
    counts and the class histogram and compared against what the resumed checkpoint was
    trained on. Continuing on top of different data is not a warning."""
    d = C.run_dir(cfg["run_name"])
    mf = d / "reports" / "manifest.json"
    m = {"fingerprint": C.fingerprint(cfg),
         "cfg": {k: v for k, v in cfg.items()},
         "class_names": list(class_names),
         "n_clients": len(spans), "n_train": sum(h - l for l, h in spans.values()),
         "client_rows": {str(c): spans[c][1] - spans[c][0] for c in sorted(spans)},
         "n_params": N_PARAMS, "select_metric": SELECT_METRIC,
         "torch": torch.__version__, "cuda": torch.version.cuda,
         "written": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}
    if extra: m.update(extra)

    # A handoff bundle carries the ids of the data its rounds were trained on; a resumed
    # session has no earlier manifest, so this is where that gate fires for it.
    old = C.read_handoff(d) if not mf.is_file() else None
    if old is not None:
        for k in ("content_id", "data_id"):
            a, b = old.get(k), m.get(k)
            if a is not None and b is not None and a != b:
                raise RuntimeError(
                    f"{k} changed: the handoff's rounds were trained on {a}, the data "
                    f"mounted now is {b}. Resuming across that is not a continuation.")
    if mf.is_file():
        old = json.loads(mf.read_text())
        for k in ("content_id", "data_id"):
            a, b = old.get(k), m.get(k)
            if a is not None and b is not None and a != b:
                raise RuntimeError(
                    f"{k} changed: this run's checkpoints were trained on {a}, the data "
                    f"mounted now is {b}. Resuming across that is not a continuation.")
        m["sessions"] = int(old.get("sessions", 1)) + 1
        m["first_written"] = old.get("first_written", old.get("written"))
    else:
        m["sessions"], m["first_written"] = 1, m["written"]

    # y_true travels with the run: a downloaded run directory must be able to check its
    # own predictions against its own confusion matrices.
    if y_true_src is not None:
        dst = d / "reports" / "y_true.u8.npy"
        if not dst.is_file():
            shutil.copyfile(y_true_src, dst)
        m["y_true"] = "reports/y_true.u8.npy"
    mf.write_text(json.dumps(m, indent=2))
    return mf


In [ ]:
%%writefile /kaggle/working/proj/verify.py
"""Re-derive every published number from the artifacts on disk.

Nothing here trusts a number because it was printed once. Per client and per round, the
10 metrics and the per-class block are recomputed from that client's confusion matrix; the
mean/std/min/max over clients are recomputed from the per-client values; the server
aggregate's 10 metrics are recomputed from its own matrix; predictions, where stored,
rebuild the confusion matrices; the client logs are checked against the step arithmetic
and the schedule they claim; and all of it is compared against the copies in the weights
file, history.csv and clients.csv.

PerFed-SKD adds one class of check the sibling rebuilds have no need for: the DEVICE
SELECTION is itself a result. tau is recomputed as the mean of the clients' accuracies
taken from their confusion matrices, S_{t+1} is recomputed by applying `select_clients`
to it, and round t's stored S_t must equal round t-1's stored S_{t+1}. Without that last
link a run could change which devices were selected between two rounds and every other
number would still reconcile.

Every check exists because its absence lets a specific tampered fixture pass: a deleted
history row, a duplicated one, a metric set to NaN (`abs(nan) > tol` is False), a per-class
F1 of 999, deleted logs, a client missing from the log, a proxy metric overwritten, a
resume file overwritten with garbage, and a config claiming 999 test rows.
"""
import csv, json, math
from pathlib import Path
import numpy as np
import torch

from proj.metrics import metrics_from_confusion, per_class_from_confusion, METRIC_KEYS
from proj.perfedskd import (expected_steps, lr_at, threshold, select_clients,
                            ACC_KEYS, SELECT_METRIC)
from proj import ckpt as C

TOL = 1e-12          # both sides come from the same float64 code path on the same counts
CSV_TOL = 1e-9       # history.csv round-trips through str()


def _finite(x):
    try: return math.isfinite(float(x))
    except (TypeError, ValueError): return False


def _read_csv(p):
    with open(p) as f:
        return list(csv.DictReader(f))


def verify_run(run_dir, cfg=None, build_model=None, expect_params=None, y_true_path=None,
               require_rounds=None, full=True):
    """Returns (ok, lines). full=True is the acceptance mode: client logs must be present
    for every round and predictions for every round that claims them."""
    d = Path(run_dir)
    fp = C.fingerprint(cfg) if cfg is not None else None
    last = C.last_complete_round(d, fp) or 0
    # A tree that starts past round 1 is a session resumed from a handoff bundle: rounds
    # before `first` are attested by reports/handoff.json (checked inside round_ok) and are
    # re-verified in full only after the sessions are merged. Nothing here is skipped for
    # the rounds that ARE on disk.
    first = C._first_round(d) if last else 1
    mode = "full" if full else "minimal"
    out = [f"run      : {d}", f"complete : rounds {first}..{last}   (mode: {mode})"
           + (f"   [handoff bundle: rounds 1..{first} attested by {C.HANDOFF}]" if first > 1 else "")]
    bad = []

    mf = d / "reports" / "manifest.json"
    man = json.loads(mf.read_text()) if mf.is_file() else None
    if man is None and full:
        bad.append("reports/manifest.json missing: the run does not describe itself")

    N = int(cfg["n_clients"]) if cfg else (int(man["n_clients"]) if man else None)
    n_test = int(cfg["n_test"]) if cfg and "n_test" in cfg else None
    want_rows = ({int(k): int(v) for k, v in man["client_rows"].items()}
                 if man and "client_rows" in man else None)

    # ---- history.csv / clients.csv: exactly rounds 1..last, once each
    hist, hp = {}, d / "history.csv"
    if hp.is_file():
        rows = _read_csv(hp)
        seen = [int(r["round"]) for r in rows]
        if len(seen) != len(set(seen)):
            bad.append(f"history.csv has duplicate rows for round(s) "
                       f"{sorted({r for r in seen if seen.count(r) > 1})}")
        if sorted(set(seen)) != list(range(first, last + 1)):
            bad.append(f"history.csv covers rounds {sorted(set(seen))}, expected {first}..{last}")
        hist = {int(r["round"]): r for r in rows}
    elif last:
        bad.append("history.csv missing")
    crows, cp = {}, d / "clients.csv"
    if cp.is_file():
        for r in _read_csv(cp):
            crows.setdefault(int(r["round"]), {})[int(r["cid"])] = r
    elif last:
        bad.append("clients.csv missing")

    y_true = None
    if y_true_path is not None and Path(y_true_path).is_file():
        y_true = np.load(y_true_path, mmap_mode="r")

    prev_next = None       # S_{t+1} committed by the previous round, for the chain check
    for r in range(first, last + 1):
        tag = f"round {r:03d}"
        cm = np.load(d / "confusion" / f"round_{r:03d}.npy")
        gcm = np.load(d / "confusion" / f"global_{r:03d}.npy")
        if cm.ndim != 3 or cm.shape[1] != cm.shape[2] or (N is not None and cm.shape[0] != N):
            bad.append(f"{tag}: confusion is {cm.shape}, expected ({N}, C, C)"); continue
        if gcm.shape != cm.shape[1:]:
            bad.append(f"{tag}: global confusion is {gcm.shape}, expected {cm.shape[1:]}"); continue
        if (cm < 0).any() or (gcm < 0).any():
            bad.append(f"{tag}: negative counts in a confusion matrix")
        if cfg and cm.shape[1] != int(cfg["num_classes"]):
            bad.append(f"{tag}: {cm.shape[1]} classes, cfg says {cfg['num_classes']}")
        totals = cm.sum(axis=(1, 2))
        if n_test is None: n_test = int(totals[0])
        if not (totals == n_test).all() or gcm.sum() != n_test:
            bad.append(f"{tag}: confusion totals {sorted(set(totals.tolist()))} / global "
                       f"{int(gcm.sum())} != n_test {n_test}")

        js = json.loads((d / "metrics" / f"round_{r:03d}.json").read_text())
        ck = torch.load(d / "weights" / f"round_{r:03d}.pt", map_location="cpu",
                        weights_only=True, mmap=True)
        rs = torch.load(d / "resume" / f"round_{r:03d}.pt", map_location="cpu",
                        weights_only=True)
        clients = js.get("clients", [])
        if [c.get("cid") for c in clients] != list(range(cm.shape[0])):
            bad.append(f"{tag}: metrics json lists clients "
                       f"{[c.get('cid') for c in clients][:5]}..., expected 0..{cm.shape[0]-1}")
            continue
        per = {k: [] for k in METRIC_KEYS}
        for c in clients:
            cid = int(c["cid"])
            rec = metrics_from_confusion(cm[cid])
            for k in METRIC_KEYS:
                v = c.get(k)
                if not _finite(v) or abs(float(v) - rec[k]) > TOL:
                    bad.append(f"{tag}: client {cid} {k} {v!r} != {rec[k]} recomputed")
                per[k].append(rec[k])
                cv = crows.get(r, {}).get(cid, {}).get(k)
                if crows and (cv is None or not _finite(cv) or abs(float(cv) - rec[k]) > CSV_TOL):
                    bad.append(f"{tag}: clients.csv client {cid} {k} = {cv!r} != {rec[k]}")
            names = [e.get("class") for e in c.get("per_class", [])]
            want_pc = per_class_from_confusion(cm[cid], names) if len(names) == cm.shape[1] else None
            if want_pc is None:
                bad.append(f"{tag}: client {cid} per-class block missing or wrong length")
            else:
                for got, want in zip(c["per_class"], want_pc):
                    for f in ("idx", "support"):
                        if int(got.get(f, -1)) != int(want[f]):
                            bad.append(f"{tag}: client {cid} class {want['idx']} {f} "
                                       f"{got.get(f)} != {want[f]}")
                    for f in ("precision", "recall", "f1"):
                        v = got.get(f)
                        if not _finite(v) or abs(float(v) - want[f]) > TOL:
                            bad.append(f"{tag}: client {cid} class {want['idx']} {f} "
                                       f"{v!r} != {want[f]}")
        # ---- the aggregate row: mean/std/min/max over clients, in json, weights, csv
        for k in METRIC_KEYS:
            v = np.asarray(per[k], dtype=np.float64)
            want = {k: float(v.mean()), f"{k}_std": float(v.std()),
                    f"{k}_min": float(v.min()), f"{k}_max": float(v.max())}
            for kk, wv in want.items():
                for where, val, tol in (("json", js.get(kk), TOL),
                                        ("history.csv", hist.get(r, {}).get(kk), CSV_TOL)):
                    if val is None:
                        bad.append(f"{tag}: {kk} missing in {where}"); continue
                    if not _finite(val):
                        bad.append(f"{tag}: {kk} in {where} is not finite ({val!r})")
                    elif abs(float(val) - wv) > tol:
                        bad.append(f"{tag}: {kk} in {where} = {val} != {wv} recomputed")
            wv = ck.get("metrics", {}).get(k)
            if wv is None or not _finite(wv) or abs(float(wv) - want[k]) > TOL:
                bad.append(f"{tag}: weights file {k} = {wv!r} != {want[k]}")
        # ---- the server aggregate: its own 10 metrics + per-class, in json, weights, csv
        grec = metrics_from_confusion(gcm)
        gj = js.get("global", {})
        for k in METRIC_KEYS:
            for where, val, tol in (("json global", gj.get(k), TOL),
                                    ("json row", js.get(f"global_{k}"), TOL),
                                    ("history.csv", hist.get(r, {}).get(f"global_{k}"), CSV_TOL),
                                    ("weights file", ck.get("global_metrics", {}).get(k), TOL)):
                if val is None:
                    bad.append(f"{tag}: global {k} missing in {where}"); continue
                if not _finite(val):
                    bad.append(f"{tag}: global {k} in {where} is not finite ({val!r})")
                elif abs(float(val) - grec[k]) > tol:
                    bad.append(f"{tag}: global {k} in {where} = {val} != {grec[k]} recomputed")
        names = [e.get("class") for e in gj.get("per_class", [])]
        if len(names) != cm.shape[1]:
            bad.append(f"{tag}: global per-class block missing or wrong length")
        else:
            for got, want in zip(gj["per_class"], per_class_from_confusion(gcm, names)):
                for f in ("precision", "recall", "f1"):
                    v = got.get(f)
                    if not _finite(v) or abs(float(v) - want[f]) > TOL:
                        bad.append(f"{tag}: global class {want['idx']} {f} {v!r} != {want[f]}")
                if int(got.get("support", -1)) != int(want["support"]):
                    bad.append(f"{tag}: global class {want['idx']} support mismatch")
        if int(js.get("evaluated", -1)) != cm.shape[0]:
            bad.append(f"{tag}: json 'evaluated' {js.get('evaluated')} != {cm.shape[0]} clients")
        if cfg is not None:
            want_lr = lr_at(cfg, r)
            if not _finite(js.get("lr")) or abs(float(js["lr"]) - want_lr) > 1e-12:
                bad.append(f"{tag}: json lr {js.get('lr')!r} != schedule {want_lr}")

        # ---- device selection, recomputed from the confusion matrices themselves
        acc_by_cid = {cid: metrics_from_confusion(cm[cid])[SELECT_METRIC]
                      for cid in range(cm.shape[0])}
        want_tau = threshold(acc_by_cid)
        want_next = select_clients(acc_by_cid, want_tau)
        for where, val in (("json", js.get("tau")), ("weights file", ck.get("tau")),
                           ("history.csv", hist.get(r, {}).get("tau"))):
            tol = CSV_TOL if where == "history.csv" else TOL
            if val is None:
                bad.append(f"{tag}: tau missing in {where}"); continue
            if not _finite(val) or abs(float(val) - want_tau) > tol:
                bad.append(f"{tag}: tau in {where} = {val!r} != {want_tau} recomputed "
                           f"as the mean of the clients' {SELECT_METRIC}")
        got_next = js.get("selected_next")
        for where, val in (("json", js.get("selected_next")),
                           ("weights file", ck.get("selected_next")),
                           ("resume file", rs.get("selected_next"))):
            if val is None:
                bad.append(f"{tag}: selected_next missing in {where}"); continue
            if [int(x) for x in val] != want_next:
                bad.append(f"{tag}: selected_next in {where} has {len(val)} devices, "
                           f"recomputing the rule a_m < tau gives {len(want_next)}")
        got_sel = js.get("selected")
        if got_sel is None:
            bad.append(f"{tag}: metrics json does not record which devices were selected")
        else:
            got_sel = [int(x) for x in got_sel]
            if [int(x) for x in ck.get("selected", [])] != got_sel:
                bad.append(f"{tag}: weights file and metrics json disagree on S_t")
            if int(js.get("n_selected", -1)) != len(got_sel):
                bad.append(f"{tag}: n_selected {js.get('n_selected')} != |S_t| {len(got_sel)}")
            if r == first:
                # The first round on disk: round 1 of a fresh run selects everyone; a
                # handoff bundle's first round inherits S_t from a session not present here.
                if first == 1 and got_sel != list(range(cm.shape[0])):
                    bad.append(f"{tag}: round 1 must select all {cm.shape[0]} devices, "
                               f"got {len(got_sel)}")
            elif prev_next is not None and got_sel != prev_next:
                bad.append(f"{tag}: S_t is not the S_{{t+1}} that round {r-1} committed "
                           f"({len(got_sel)} vs {len(prev_next)} devices)")
            if int(js.get("n_selected_next", -1)) != len(want_next):
                bad.append(f"{tag}: n_selected_next {js.get('n_selected_next')} != "
                           f"{len(want_next)}")
        prev_next = [int(x) for x in got_next] if got_next is not None else None

        if fp is not None and ck.get("fingerprint") != fp:
            bad.append(f"{tag}: weights fingerprint {ck.get('fingerprint')} != {fp}")
        if build_model is not None:
            try:
                C.load_weights(d / "weights" / f"round_{r:03d}.pt", build_model, expect_params)
            except Exception as e:
                bad.append(f"{tag}: weights do not rebuild the models: {e}")

        # ---- predictions tie the matrices back to model output, where stored
        pp = d / "preds" / f"round_{r:03d}.u8.npy"
        gp = d / "preds" / f"global_{r:03d}.u8.npy"
        claims = cfg is not None and r in set(cfg.get("preds_rounds", [cfg["rounds"]]))
        if pp.is_file():
            yp = np.load(pp, mmap_mode="r")
            ygp = np.load(gp, mmap_mode="r") if gp.is_file() else None
            if yp.shape != (cm.shape[0], n_test):
                bad.append(f"{tag}: predictions are {yp.shape}, expected ({cm.shape[0]}, {n_test})")
            elif ygp is None or ygp.shape != (n_test,):
                bad.append(f"{tag}: global predictions missing or not ({n_test},)")
            elif y_true is not None:
                if len(y_true) != n_test:
                    bad.append(f"{tag}: y_true has {len(y_true)} rows, test has {n_test}")
                else:
                    k = cm.shape[1]
                    yt = np.asarray(y_true, np.int64) * k
                    for cid in range(cm.shape[0]):
                        rebuilt = np.bincount(yt + np.asarray(yp[cid], np.int64),
                                              minlength=k * k).reshape(k, k)
                        if not (rebuilt == cm[cid]).all():
                            bad.append(f"{tag}: client {cid} confusion != stored predictions")
                    rebuilt = np.bincount(yt + np.asarray(ygp, np.int64),
                                          minlength=k * k).reshape(k, k)
                    if not (rebuilt == gcm).all():
                        bad.append(f"{tag}: global confusion != stored predictions")
            elif full:
                bad.append(f"{tag}: predictions present but no y_true to check them against")
        elif claims and full:
            bad.append(f"{tag}: cfg claims predictions for this round but none are stored")

        # ---- client logs: every client, step arithmetic and the schedule have to close
        lp = d / "logs" / f"round_{r:03d}.json"
        if not lp.is_file():
            if full:
                bad.append(f"{tag}: no client log; participation is unattested")
        else:
            try: lg = json.loads(lp.read_text())
            except Exception as e:
                bad.append(f"{tag}: client log unreadable: {e}"); lg = {}
            if got_sel is not None and [int(x) for x in lg.get("selected", [])] != got_sel:
                bad.append(f"{tag}: client log's `selected` differs from the metrics json")
            got_ids = sorted(int(e["cid"]) for e in lg.get("clients", []))
            if got_ids != list(range(cm.shape[0])):
                bad.append(f"{tag}: log has clients {got_ids[:6]}..., expected all "
                           f"0..{cm.shape[0] - 1}")
            for e in lg.get("clients", []):
                c = e.get("cid")
                if e.get("applied", 0) + e.get("skipped", 0) != e.get("steps", -1):
                    bad.append(f"{tag}: client {c} applied+skipped != steps")
                if e.get("applied", 0) <= 0:
                    bad.append(f"{tag}: client {c} applied no step")
                if e.get("nonfinite", 0):
                    bad.append(f"{tag}: client {c} applied a step with a non-finite gradient")
                if cfg and "n_k" in e:
                    s = expected_steps(int(e["n_k"]), cfg)
                    if int(e.get("steps", -1)) != s:
                        bad.append(f"{tag}: client {c} ran {e.get('steps')} steps, expected {s}")
                if want_rows is not None and c in want_rows and int(e.get("n_k", -1)) != want_rows[c]:
                    bad.append(f"{tag}: client {c} trained on {e.get('n_k')} rows, "
                               f"manifest says {want_rows[c]}")
                # The rate the client actually used must be the schedule's value for this
                # round: a resumed session that planned a different horizon would otherwise
                # continue the run at a rate the fingerprint never saw.
                if cfg is not None:
                    want_lr = lr_at(cfg, r)
                    if not _finite(e.get("lr")) or abs(float(e["lr"]) - want_lr) > 1e-12:
                        bad.append(f"{tag}: client {c} trained at lr {e.get('lr')!r}, "
                                   f"schedule says {want_lr}")
                if got_sel is not None and bool(e.get("selected")) != (c in set(got_sel)):
                    bad.append(f"{tag}: client {c} log says selected={e.get('selected')}, "
                               f"the round's subset says {c in set(got_sel)}")
                for f in ACC_KEYS:
                    if not _finite(e.get(f)):
                        bad.append(f"{tag}: client {c} {f} is not finite ({e.get(f)!r})")

    n_preds = len(list((d / "preds").glob("round_*.u8.npy"))) if (d / "preds").is_dir() else 0
    n_logs = len(list((d / "logs").glob("round_*.json"))) if (d / "logs").is_dir() else 0
    out.append(f"artifacts: {n_preds} prediction files, {n_logs} client logs"
               + ("" if y_true is not None else "   (predictions NOT cross-checked: no y_true)"))
    if require_rounds is not None and last != require_rounds:
        bad.append(f"run is INCOMPLETE: {last} of {require_rounds} rounds")
    out += [f"  FAIL {b}" for b in bad] or ["  all artifact checks passed"]
    return not bad, out


In [ ]:
import wandb
# BEGIN INLINE WANDB CREDENTIAL — owner-authorized private notebook
wandb.login(key=__import__("os").environ["WANDB_API_KEY"], relogin=True, verify=True)
# END INLINE WANDB CREDENTIAL
# W&B authenticates before dataset decode; inline key use was explicitly authorized by the owner.
run = wandb.init(project="perfedskd-veremi", name="perfedskd_100c_probe", config=CFG,
                 resume="allow", id="perfedskd_100c_probe")
print("W&B:", run.url)


In [ ]:
# Cheap identity work, then the resume gate, then the decode. A continuation push must
# die at the gate, not after a two-minute parquet pass.
import hashlib, json, time, numpy as np
from pathlib import Path
from proj import ckpt as C
from proj.data import (find_root, load_clients, load_test, assert_fp16_safe,
                       cache_ok)

FL_ROOT = find_root("train/client_id=000")
CEN_ROOT = find_root("upload/test")
TEST_ROOT = CEN_ROOT / "upload/test"
SCALER = json.loads((CEN_ROOT / "upload/scaler.json").read_text())["features"]
assert len(SCALER) == 66, f"scaler has {len(SCALER)} entries, expected 66"
FEATS = ['f_rcv_pos_noise_x', 'f_rcv_pos_noise_y', 'f_rcv_spd', 'f_rcv_spd_noise', 'f_rcv_acl', 'f_rcv_acl_noise', 'f_rcv_hed_noise', 'f_snd_pos_noise_x', 'f_snd_pos_noise_y', 'f_snd_spd', 'f_snd_spd_noise', 'f_snd_acl', 'f_snd_acl_noise', 'f_snd_hed_noise', 'f_snd_dist_road_edge', 'f_rcv_x_rel', 'f_rcv_y_rel', 'f_snd_x_rel', 'f_snd_y_rel', 'f_delay_s', 'f_dx', 'f_dy', 'f_dist', 'f_bearing_sin', 'f_bearing_cos', 'f_rcv_hed_sin', 'f_rcv_hed_cos', 'f_snd_hed_sin', 'f_snd_hed_cos', 'f_hed_diff_cos', 'f_rcv_vx', 'f_rcv_vy', 'f_snd_vx', 'f_snd_vy', 'f_rel_speed', 'f_closing_speed', 'f_spd_diff', 'f_rcv_noise_mag', 'f_snd_noise_mag', 'f_first_in_session', 'f_sess_idx', 'f_sess_dt', 'f_sess_dt_send', 'f_sess_dt_skew', 'f_sess_dpos', 'f_sess_implied_spd', 'f_sess_spd_residual', 'f_sess_dspd', 'f_sess_acl_residual', 'f_sess_dhed', 'f_sess_dmsgid', 'f_sess_ddist', 'f_sess_ddre', 'f_sess_pos_pred_err', 'f_alias_age_s', 'f_rx_rate_1s', 'f_rx_rate_5s', 'f_sender_rate_1s', 'f_sender_rate_5s', 'f_sender_share_5s', 'f_rcv_profile_normal', 'f_rcv_profile_cautious', 'f_rcv_profile_aggressive', 'f_snd_profile_normal', 'f_snd_profile_cautious', 'f_snd_profile_aggressive']
CLASS_NAMES = ['benign', 'accelerationMultiplication', 'constantPositionOffset', 'constantSpeedOffset', 'dataReplay', 'dosAttack', 'feignedBraking', 'positionMirroring', 'randomPositionOffset', 'randomSpeedOffset', 'reversedHeading', 'suddenConstantSpeed', 'suddenStop', 'timeDelayAttack', 'trafficCongestionSybil', 'zeroSpeedReport']

# What the fingerprint could not otherwise see: a permuted feature order, a re-fitted
# scaler or a different partition keep every shape identical.
CFG["data_id"] = hashlib.sha256(json.dumps({
    "features": FEATS, "classes": CLASS_NAMES, "n_clients": CFG["n_clients"],
    "scaler": [[SCALER[c]["mean"], SCALER[c]["std_used"]] for c in FEATS],
}, sort_keys=True).encode()).hexdigest()[:16]
print("FL root    :", FL_ROOT)
print("test root  :", TEST_ROOT)
print("data_id    :", CFG["data_id"])
print("fingerprint:", C.fingerprint(CFG))

last = C.resolve_resume(CFG["run_name"], CFG)
if CFG["require_resume"] and last is None:
    raise SystemExit("require_resume set but no VERIFIED checkpoint found — fix the "
                     "attachment. A marker without its artifacts does not count.")
print("resume from round", last)

cache = Path(CFG["cache"]); cache.mkdir(parents=True, exist_ok=True)
MF = cache / "manifest.json"
FILES = ("train_X.f16.npy", "train_y.u8.npy", "test_X.f16.npy", "test_y.u8.npy",
         "spans.json")
want = {"data_id": CFG["data_id"], "n_clients": CFG["n_clients"],
        "fl_root": str(FL_ROOT), "test_root": str(TEST_ROOT)}

t0 = time.time()
if cache_ok(cache, want, CFG["n_clients"]):
    print("prepack cache reusable (manifest matches and every file checks out)")
else:
    for f in FILES: (cache / f).unlink(missing_ok=True)
    MF.unlink(missing_ok=True)
    X, Y, spans = load_clients(FL_ROOT, FEATS, CFG["n_clients"])
    print(f"train {X.shape} max|x|={assert_fp16_safe(X,'train'):.1f}")
    np.save(cache / "train_X.f16.npy", X); np.save(cache / "train_y.u8.npy", Y)
    json.dump({str(k): v for k, v in spans.items()}, open(cache / "spans.json", "w"))
    del X, Y
    TX, TY = load_test(TEST_ROOT, FEATS, SCALER)
    print(f"test  {TX.shape} max|x|={assert_fp16_safe(TX,'test'):.1f}")
    np.save(cache / "test_X.f16.npy", TX); np.save(cache / "test_y.u8.npy", TY)
    del TX, TY
    MF.write_text(json.dumps(want))                       # cache marker: absolutely last
    assert cache_ok(cache, want, CFG["n_clients"]), "the cache just written does not validate"

spans = {int(k): tuple(v) for k, v in json.load(open(cache / "spans.json")).items()}
CFG["n_test"] = len(np.load(cache / "test_y.u8.npy", mmap_mode="r"))
n_train = sum(h - l for l, h in spans.values())
assert n_train == 43_045_415, f"train rows {n_train} != 43,045,415"
assert CFG["n_test"] == 10_761_343, f"test rows {CFG['n_test']}"
assert len(spans) == CFG["n_clients"], f"{len(spans)} spans for {CFG['n_clients']} clients"

# content_id reads the labels that are actually cached, hit or miss: the row counts and
# the class histogram change when the partition or the file contents change.
_ytr = np.load(cache / "train_y.u8.npy", mmap_mode="r")
_yte = np.load(cache / "test_y.u8.npy", mmap_mode="r")
assert len(_ytr) == n_train, f"train X/y disagree: {len(_ytr)} labels for {n_train} rows"
CFG["content_id"] = hashlib.sha256(json.dumps({
    "clients": [[c, spans[c][0], spans[c][1]] for c in sorted(spans)],
    "train_hist": np.bincount(np.asarray(_ytr), minlength=16).tolist(),
    "test_hist": np.bincount(np.asarray(_yte), minlength=16).tolist(),
}, sort_keys=True).encode()).hexdigest()[:16]
del _ytr, _yte
print("content_id :", CFG["content_id"])
print(f"prepack {time.time()-t0:.1f}s | {n_train:,} train / {CFG['n_test']:,} test rows")


In [ ]:
# ---- PROBE ONLY: measure what sm_86 cannot tell us about the T4 before the rounds run.
# Train: one client's SKD epoch (frozen-teacher forward + student forward/backward per
# step) eager vs compiled. Eval: one folded model over the FULL test set, eager vs
# compiled, at two batch sizes -- eval is 45-65 % of a round here, so its batch is the
# single most valuable number this cell produces. Everything is freed afterwards so the
# two workers start with the whole GPU.
import gc, time, torch
from proj.model import build_model
from proj.perfedskd import (layout, flatten, unflatten_into, client_update,
                            make_optimizer, ACC_KEYS)
from proj.evaluate import fold_bn, load_folded, eval_model
from proj import driver as D

CAL = {}
dev = torch.device("cuda:0"); torch.cuda.set_device(dev)
torch.backends.cudnn.benchmark = True
for _n in ('recompile_limit', 'cache_size_limit'):
    if hasattr(torch._dynamo.config, _n): setattr(torch._dynamo.config, _n, 64)
cache = Path(CFG["cache"])
TX = D._resident(cache / "test_X.f16.npy", dev); TY = D._resident(cache / "test_y.u8.npy", dev)
lo, hi = spans[min(spans, key=lambda c: spans[c][1] - spans[c][0])]   # smallest client
hi = min(hi, lo + 400 * CFG["batch"])                                  # ~400 steps
X = torch.from_numpy(np.load(cache / "train_X.f16.npy", mmap_mode="r")[lo:hi].copy()).to(dev)
Y = torch.from_numpy(np.load(cache / "train_y.u8.npy", mmap_mode="r")[lo:hi].copy()).to(dev)

def one_client(Sc, Se, Vc, Ve):
    opt = make_optimizer(Se, CFG["lr"], CFG, fused=True)
    sc = torch.amp.GradScaler("cuda"); sc.scale(torch.zeros(1, device=dev))
    g = torch.Generator(device=dev); g.manual_seed(1)
    torch.cuda.synchronize(); t0 = time.perf_counter()
    acc, n = client_update(Sc, Se, Vc, Ve, opt, sc, X, Y, 0, hi - lo, CFG, g)
    torch.cuda.synchronize()
    return (time.perf_counter() - t0) / n * 1000, n, int(acc[len(ACC_KEYS)].item())

torch.manual_seed(CFG["seed"])
Se = build_model(CFG).to(dev).train(); Ve = build_model(CFG).to(dev).eval()
for _p in Ve.parameters(): _p.requires_grad_(False)
fk, ik, _ = layout(Se)
s0, i0 = flatten(Se, fk, ik); v0, vi0 = flatten(Ve, fk, ik)
def reset():
    unflatten_into(Se, s0, i0, fk, ik); unflatten_into(Ve, v0, vi0, fk, ik)
ms, n, sk = one_client(Se, Se, Ve, Ve)
CAL["train_eager_ms_per_step"] = ms
print(f"train eager   : {ms:.2f} ms per step ({n} steps, {sk} skipped) at batch {CFG['batch']}")
reset()
t0 = time.perf_counter()
Sc, Vc = D._compile_train(Se, Ve, CFG, dev, X[:CFG["batch"]].float(), Y[:CFG["batch"]].long())
CAL["compile_seconds"] = time.perf_counter() - t0
CAL["train_backend"] = "compiled" if Sc is not Se else "eager"
if Sc is not Se:
    reset(); one_client(Sc, Se, Vc, Ve)                          # warm the graphs
    reset(); ms, n, sk = one_client(Sc, Se, Vc, Ve)
    CAL["train_compiled_ms_per_step"] = ms
    print(f"train compiled: {ms:.2f} ms per step ({n} steps, {sk} skipped) | "
          f"{CAL['train_eager_ms_per_step']/ms:.2f}x | compile+gate {CAL['compile_seconds']:.0f}s")
del Sc, Vc

Te = fold_bn(build_model(CFG).to(dev)); load_folded(Te, Se)
for eb in (16384, 32768):
    c = dict(CFG, eval_batch=eb)
    torch.cuda.synchronize(); t0 = time.perf_counter()
    cm, nf, _ = eval_model(Te, Te, TX, TY, c); torch.cuda.synchronize()
    r = TX.shape[0] / (time.perf_counter() - t0); CAL[f"eval_eager_folded_{eb}"] = r
    print(f"eval eager-folded  batch {eb:>5}: {r:,.0f} rows/s")
    if CFG["compile"]:
        Tc = D._compile_eval(Te, c, dev, TX[:eb].float())
        if Tc is not Te:
            eval_model(Tc, Te, TX[:4 * eb], TY[:4 * eb], c)          # warm
            torch.cuda.synchronize(); t0 = time.perf_counter()
            cm2, nf2, _ = eval_model(Tc, Te, TX, TY, c); torch.cuda.synchronize()
            r = TX.shape[0] / (time.perf_counter() - t0); CAL[f"eval_compiled_folded_{eb}"] = r
            print(f"eval compiled-folded batch {eb:>5}: {r:,.0f} rows/s | "
                  f"|dCM|={int((cm2 - cm).abs().sum())} cells of {TX.shape[0]}")
            torch._dynamo.reset()
        del Tc
# What a full round would cost from these two numbers alone, before any round has run.
_mt = CAL.get("train_compiled_ms_per_step", CAL["train_eager_ms_per_step"]) / 1000.0
_re = max(CAL.get("eval_compiled_folded_16384", 0), CAL.get("eval_eager_folded_16384", 1))
_steps = sum(-(-(h - l) // CFG["batch"]) for l, h in spans.values()) * CFG["local_epochs"]
CAL["projected_train_sec"] = _steps * _mt / CFG["world_size"]
CAL["projected_eval_sec"] = (CFG["n_clients"] + 1) / CFG["world_size"] * TX.shape[0] / _re
CAL["projected_round_sec"] = CAL["projected_train_sec"] + CAL["projected_eval_sec"]
print(f"projected round: train {CAL['projected_train_sec']:.0f}s "
      f"({_steps:,} steps / {CFG['world_size']} GPUs) + eval {CAL['projected_eval_sec']:.0f}s "
      f"({CFG['n_clients']}+1 models) = {CAL['projected_round_sec']/60:.1f} min; "
      f"{CFG['rounds']} rounds = {CAL['projected_round_sec']*CFG['rounds']/3600:.1f} h")
del Te, Se, Ve, X, Y, TX, TY, s0, v0
gc.collect(); torch.cuda.empty_cache(); torch._dynamo.reset()
CAL["vram_after_free_gb"] = torch.cuda.memory_allocated(dev) / 2**30
print("calibration:", json.dumps({k: (round(v, 3) if isinstance(v, float) else v) for k, v in CAL.items()}))
(C.run_dir(CFG["run_name"]) / "reports" / "calibration.json").write_text(json.dumps(CAL, indent=1))
if run is not None:
    run.summary.update({f"cal_{k}": v for k, v in CAL.items()})


In [ ]:
from proj.driver import run as train, write_manifest
# Raises if this run's checkpoints were trained on different data. The resume gate above
# ran before the decode and could only compare data_id; content_id is the post-decode one.
write_manifest(CFG, CLASS_NAMES, spans,
               y_true_src=Path(CFG["cache"]) / "test_y.u8.npy",
               extra={"fl_root": str(FL_ROOT), "test_root": str(TEST_ROOT),
                      "feature_cols": FEATS, "n_test": CFG["n_test"],
                      "data_id": CFG["data_id"], "content_id": CFG["content_id"],
                      "scaler": {c: [SCALER[c]["mean"], SCALER[c]["std_used"]]
                                 for c in FEATS}})
hist = train(CFG, spans, CLASS_NAMES, wandb_run=run, t_origin=T0)
print(f"\ncompleted {len(hist)} rounds this session")


In [ ]:
# Every published number, re-derived from the artifacts on disk. Never from memory.
import csv
from proj.verify import verify_run
from proj.model import build_model, N_PARAMS
from proj.metrics import METRIC_KEYS

d = C.run_dir(CFG["run_name"])
# y_true from the RUN, not from /kaggle/temp: the cache is gone with the session, and the
# check has to be the same one someone can repeat after downloading the output alone.
ok, lines = verify_run(d, cfg=CFG, build_model=build_model, expect_params=N_PARAMS,
                       y_true_path=d / "reports" / "y_true.u8.npy", full=True)
print("\n".join(lines))

last = C.last_complete_round(d, C.fingerprint(CFG)) or 0
print(f"\nrounds verified : {last} / {CFG['rounds']}")
if last < CFG["rounds"]:
    print(f"  INCOMPLETE — attach this notebook's output (or a checkpoint dataset of it) to "
          f"the next push and regenerate with --require-resume to continue at round {last + 1}")
rows = [r for r in csv.DictReader(open(d / "history.csv")) if int(r["round"]) <= last]
if rows:
    fin = rows[-1]
    # The headline is the LAST round, fixed before the run. best-f1 is chosen on the test
    # set after seeing it, so it is a description of the curve and not a second result.
    print(f"\nresult at round {fin['round']} (mean over {CFG['n_clients']} personalized models; "
          f"std / min / max of f1_macro {float(fin['f1_macro_std']):.4f} / "
          f"{float(fin['f1_macro_min']):.4f} / {float(fin['f1_macro_max']):.4f}):")
    for k in METRIC_KEYS: print(f"  {k:<20} {float(fin[k]):.6f}   server aggregate {float(fin['global_' + k]):.6f}")
    sel = [int(r["n_selected"]) for r in rows]
    print(f"\nselection: |S| per round min {min(sel)} median {sorted(sel)[len(sel)//2]} "
          f"max {max(sel)} of {CFG['n_clients']}; mean downlink+uplink saving "
          f"{100*sum(float(r['comm_saving']) for r in rows)/len(rows):.1f}% against "
          f"broadcasting to every device")
    b = max(rows, key=lambda r: float(r["f1_macro"]))
    print(f"\n[descriptive only] best mean f1_macro {float(b['f1_macro']):.6f} "
          f"at round {b['round']} — picked on test, not a reported result")

# Calibration, from THIS session's rounds only (the CSV would mix in imported rounds).
sec = [float(r["seconds"]) for r in hist]
overhead = (time.monotonic() - T0) - sum(sec)
print(f"\nbackend  : train {CFG.get('backend', '?')} | eval {CFG.get('backend_eval', '?')}")
print(f"session  : {len(hist)} round(s) here | startup+prepack+compile {overhead/60:.1f} min"
      f" | verify and W&B are outside this figure")
if len(sec) < 2:
    print("timing   : need 2 completed rounds to separate startup from steady state; "
          f"got {len(sec)}. No projection.")
else:
    steady = sum(sec[1:]) / len(sec[1:])
    vt = max(float(r.get("vram_train_gb", 0) or 0) for r in hist)
    ve = max(float(r.get("vram_eval_gb", 0) or 0) for r in hist)
    tr = sum(float(r["train_sec"]) for r in hist[1:]) / len(hist[1:])
    ev = sum(float(r["eval_sec"]) for r in hist[1:]) / len(hist[1:])
    print(f"timing   : rounds {[round(x) for x in sec[-3:]]}s | steady {steady:.0f}s/round "
          f"= train {tr:.0f}s + eval {ev:.0f}s ({hist[-1]['evaluated']} clients + aggregate) + commit")
    print(f"VRAM     : train {vt:.2f} GiB/GPU | eval {ve:.2f} GiB/GPU (of 16)")
    print(f"projected: {CFG['rounds']} rounds = "
          f"{(steady*CFG['rounds'] + overhead)/3600:.2f} h "
          f"({(steady*CFG['rounds'])/3600:.2f} h of rounds + {overhead/3600:.2f} h startup)")
if run is not None: run.finish()
assert ok, "artifact verification FAILED — see the FAIL lines above"
